In [ ]:
import os
# Configure before running. See README; use a fresh output tree.
os.environ.setdefault("EGLIMUC_WORKSPACE", "/content/eglimuc")
os.environ.setdefault("EGLIMUC_MOUNT_DRIVE", "0")


In [ ]:
# ============================================================
# CELL 1
# LIMUC ViT-B/16 Loss Screening — Development-Only Setup
# ------------------------------------------------------------
# Purpose:
#   1. Load ONLY the official LIMUC train/validation partition.
#   2. Audit image-label heterogeneity within patients.
#   3. Construct a derived patient-level reference label:
#        modal image-level MES, with lower-class tie-breaking.
#   4. Audit alternative patient-reference definitions:
#        modal-high, median-low, median-high, maximum label.
#   5. Create one frozen patient-stratified 5-fold assignment.
#   6. Define the ViT loss candidates for development-only
#      screening.
#   7. Save environment and reproducibility metadata.
#
# Important:
#   - The official LIMUC test set is NOT defined or loaded.
#   - Cell 2 verifies the generated fold/reference files.
#   - Cell 3 freezes the analysis plan and begins training.
# ============================================================


# ============================================================
# 0. Install dependencies
# ============================================================

%pip -q install timm scikit-learn pandas matplotlib pillow


# ============================================================
# 1. Imports
# ============================================================

import os
import re
import json
import random
import platform
import warnings
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

import numpy as np
import pandas as pd

from PIL import ImageFile

import torch
import torchvision
import timm

from sklearn.model_selection import StratifiedKFold

from IPython.display import display


ImageFile.LOAD_TRUNCATED_IMAGES = True
warnings.filterwarnings("ignore")


# ============================================================
# 2. Development-only safety check
# ============================================================

# These objects should not exist when the revised development-only
# screening notebook is run. If they are present, restart the runtime.
FORBIDDEN_TEST_OBJECTS = [
    "TEST_PATH",
    "test_df",
    "test_loader",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
]

present_test_objects = [
    object_name
    for object_name in FORBIDDEN_TEST_OBJECTS
    if object_name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects remain in the current runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only the revised development-only "
        "Cells 1–3."
    )

print("Development-only safety check passed.")
print("Official LIMUC test data are not defined or loaded.")


# ============================================================
# 3. Reproducibility
# ============================================================

def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


seed_everything(42)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


# ============================================================
# 4. Mount Google Drive
# ============================================================

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive

    (drive.mount(
        "/content/drive",
        force_remount=True,
    ) if os.environ.get("EGLIMUC_MOUNT_DRIVE", "0") == "1" else None)


# ============================================================
# 5. Paths
# ============================================================

# Official development partition only.
BASE_PATH = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/train_and_validation_sets')
)

# Revision output directory.
REVISION_ROOT = Path(
    (os.environ.get("EGLIMUC_WORKSPACE", "/content/eglimuc") + '/LIMUC/ViT2/revision_2026')
)

BASE_SAVE_ROOT = (
    REVISION_ROOT /
    "01_vit_loss_screening_development_only"
)

BASE_SAVE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

if not BASE_PATH.exists():
    raise FileNotFoundError(
        f"Development dataset path does not exist:\n{BASE_PATH}"
    )

print("Development dataset path:", BASE_PATH)
print("Revision output path:", BASE_SAVE_ROOT)


# ============================================================
# 6. Base experimental configuration
# ============================================================

BASE_CONFIG = {
    # General
    "project_name": "LIMUC_ViT_loss_screening_development_only",
    "seed": 42,
    "fold_seed": 42,

    # Input
    "image_size": 384,
    "batch_size": 8,
    "num_workers": 2,

    # Optimization
    "epochs": 25,
    "lr": 1e-4,
    "min_lr_ratio": 0.05,
    "warmup_epochs": 3,
    "weight_decay": 1e-4,
    "label_smoothing": 0.02,
    "grad_clip_norm": 1.0,
    "early_stop_patience": 8,

    # Model
    "num_classes": 4,
    "backbone_family": "vit",
    "model_name": "vit_base_patch16_384",
    "pretrained": True,
    "drop_rate": 0.1,
    "drop_path_rate": 0.05,

    # Cross-validation
    "num_folds": 5,

    # Training/inference
    "use_mixed_precision": True,
    "use_weighted_sampler": False,
    "use_class_weights_in_ce": True,
    "use_tta": True,
    "tta_horizontal_flip": True,

    # Patient-level benchmark definition
    "primary_patient_reference": "modal_low",
    "patient_reference_definition": (
        "Modal image-level MES; when multiple modal classes are tied, "
        "the lower class is selected."
    ),
    "patient_aggregation": "mean_probability",

    # Bootstrap uncertainty for OOF patient results
    "bootstrap_repetitions": 5000,
    "bootstrap_seed": 2026,

    # Documentation
    "official_test_loaded": False,
    "official_test_evaluated": False,
    "batch_size_note": (
        "Batch size 8 was used for 384x384 inputs because of "
        "GPU memory constraints."
    ),
}


# ============================================================
# 7. ViT loss candidates
# ============================================================

EXPERIMENT_PRESETS = {
    "vit_L1_ce_only": {
        "display_name": "CE only",
        "loss_type": "ce",
        "ordinal_loss_weight": 0.0,
        "ordinal_alpha": 1.0,
    },

    "vit_L2_expected_value_ordinal_mae_w010": {
        "display_name": (
            "CE + expected-value ordinal MAE, lambda=0.10"
        ),
        "loss_type": "ce_expected_value_mae",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 1.0,
    },

    "vit_L3_expected_ordinal_distance_a1_w010": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=1, lambda=0.10"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 1.0,
    },

    "vit_L4_expected_ordinal_distance_a2_w010": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=2, lambda=0.10"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.10,
        "ordinal_alpha": 2.0,
    },

    "vit_L5_expected_ordinal_distance_a1_w020": {
        "display_name": (
            "CE + expected ordinal-distance loss, "
            "alpha=1, lambda=0.20"
        ),
        "loss_type": "ce_expected_distance",
        "ordinal_loss_weight": 0.20,
        "ordinal_alpha": 1.0,
    },
}

# Development-only screening stage.
EXPERIMENTS_TO_RUN = list(
    EXPERIMENT_PRESETS.keys()
)

# Use one seed for initial screening.
# The top two candidates will later be confirmed using more seeds.
TRAIN_SEEDS = [42]


def validate_experiment_configuration() -> None:
    valid_loss_types = {
        "ce",
        "ce_expected_value_mae",
        "ce_expected_distance",
    }

    if BASE_CONFIG["num_classes"] != 4:
        raise ValueError(
            "This notebook expects four MES classes."
        )

    if BASE_CONFIG["num_folds"] < 2:
        raise ValueError(
            "num_folds must be at least 2."
        )

    if BASE_CONFIG["batch_size"] <= 0:
        raise ValueError(
            "batch_size must be positive."
        )

    if BASE_CONFIG["epochs"] <= 0:
        raise ValueError(
            "epochs must be positive."
        )

    if BASE_CONFIG["lr"] <= 0:
        raise ValueError(
            "lr must be positive."
        )

    if not (
        0.0 <
        BASE_CONFIG["min_lr_ratio"]
        <=
        1.0
    ):
        raise ValueError(
            "min_lr_ratio must be in (0, 1]."
        )

    for experiment_name, preset in EXPERIMENT_PRESETS.items():
        if preset["loss_type"] not in valid_loss_types:
            raise ValueError(
                f"{experiment_name} has an invalid loss_type: "
                f"{preset['loss_type']}"
            )

        if preset["ordinal_loss_weight"] < 0:
            raise ValueError(
                f"{experiment_name} has a negative ordinal weight."
            )

        if (
            preset["loss_type"] == "ce" and
            preset["ordinal_loss_weight"] != 0.0
        ):
            raise ValueError(
                f"{experiment_name}: CE-only must have "
                "ordinal_loss_weight=0."
            )

        if (
            preset["loss_type"] == "ce_expected_distance" and
            preset["ordinal_alpha"] <= 0
        ):
            raise ValueError(
                f"{experiment_name}: ordinal_alpha must be positive."
            )

    unknown_experiments = (
        set(EXPERIMENTS_TO_RUN) -
        set(EXPERIMENT_PRESETS)
    )

    if unknown_experiments:
        raise KeyError(
            "Unknown experiments in EXPERIMENTS_TO_RUN: "
            f"{sorted(unknown_experiments)}"
        )

    if len(TRAIN_SEEDS) == 0:
        raise ValueError(
            "TRAIN_SEEDS cannot be empty."
        )


validate_experiment_configuration()

print("\nViT candidates:")
for experiment_name in EXPERIMENTS_TO_RUN:
    print(
        "-",
        experiment_name,
        ":",
        EXPERIMENT_PRESETS[experiment_name]["display_name"],
    )

print("Training seeds:", TRAIN_SEEDS)


# ============================================================
# 8. Dataset parsing helpers
# ============================================================

IMG_EXTS = {
    ".bmp",
    ".png",
    ".jpg",
    ".jpeg",
}


def infer_label_from_parent_folder(
    image_path: Path,
):
    parent_name = (
        image_path
        .parent
        .name
        .lower()
        .strip()
    )

    label_map = {
        "mayo 0": 0,
        "mayo0": 0,
        "mayo_0": 0,

        "mayo 1": 1,
        "mayo1": 1,
        "mayo_1": 1,

        "mayo 2": 2,
        "mayo2": 2,
        "mayo_2": 2,

        "mayo 3": 3,
        "mayo3": 3,
        "mayo_3": 3,
    }

    return label_map.get(
        parent_name
    )


def extract_patient_id(
    filename: str,
):
    match = re.search(
        r"UC_patient_(\d+)_\d+\.(bmp|png|jpg|jpeg)$",
        filename,
        flags=re.IGNORECASE,
    )

    return (
        int(match.group(1))
        if match
        else None
    )


def extract_image_index(
    filename: str,
):
    match = re.search(
        r"UC_patient_\d+_(\d+)\.(bmp|png|jpg|jpeg)$",
        filename,
        flags=re.IGNORECASE,
    )

    return (
        int(match.group(1))
        if match
        else None
    )


def collect_development_records(
    root_path: Path,
) -> pd.DataFrame:
    rows = []

    for image_path in root_path.rglob("*"):
        if (
            image_path.is_file() and
            image_path.suffix.lower() in IMG_EXTS
        ):
            rows.append({
                "path": str(image_path),
                "relative_path": str(
                    image_path.relative_to(root_path)
                ),
                "filename": image_path.name,
                "label": infer_label_from_parent_folder(
                    image_path
                ),
                "patient_id": extract_patient_id(
                    image_path.name
                ),
                "image_index": extract_image_index(
                    image_path.name
                ),
            })

    dataframe = pd.DataFrame(
        rows
    )

    if len(dataframe) == 0:
        raise RuntimeError(
            "No development images were found."
        )

    return dataframe


# ============================================================
# 9. Load and validate the development partition
# ============================================================

trainval_df = collect_development_records(
    BASE_PATH
)

if trainval_df["label"].isna().any():
    bad_rows = trainval_df[
        trainval_df["label"].isna()
    ].head(20)

    raise RuntimeError(
        "Some development labels could not be parsed.\n"
        f"{bad_rows}"
    )

if trainval_df["patient_id"].isna().any():
    bad_rows = trainval_df[
        trainval_df["patient_id"].isna()
    ].head(20)

    raise RuntimeError(
        "Some patient IDs could not be parsed.\n"
        f"{bad_rows}"
    )

trainval_df["label"] = (
    trainval_df["label"]
    .astype(int)
)

trainval_df["patient_id"] = (
    trainval_df["patient_id"]
    .astype(int)
)

observed_labels = set(
    trainval_df["label"].unique()
)

expected_labels = {
    0,
    1,
    2,
    3,
}

if not observed_labels.issubset(
    expected_labels
):
    raise RuntimeError(
        "Unexpected MES labels were detected: "
        f"{sorted(observed_labels)}"
    )

trainval_df = (
    trainval_df
    .sort_values(
        [
            "patient_id",
            "image_index",
            "filename",
        ],
        na_position="last",
    )
    .reset_index(drop=True)
)

print("\nDevelopment images:", len(trainval_df))
print(
    "Development patients:",
    trainval_df["patient_id"].nunique(),
)

print("\nImage counts by MES:")
display(
    trainval_df["label"]
    .value_counts()
    .sort_index()
    .rename("n_images")
)


# ============================================================
# 10. Patient-reference audit helpers
# ============================================================

def shannon_entropy_base2(
    labels: np.ndarray,
) -> float:
    labels = np.asarray(
        labels,
        dtype=int,
    )

    counts = (
        pd.Series(labels)
        .value_counts()
        .sort_index()
    )

    probabilities = (
        counts.to_numpy(
            dtype=float
        ) /
        counts.sum()
    )

    entropy = -np.sum(
        probabilities *
        np.log2(probabilities)
    )

    if abs(entropy) < 1e-12:
        entropy = 0.0

    return float(
        entropy
    )


def summarize_one_patient(
    patient_id: int,
    patient_group: pd.DataFrame,
) -> dict:
    labels = (
        patient_group["label"]
        .astype(int)
        .to_numpy()
    )

    if len(labels) == 0:
        raise RuntimeError(
            f"Patient {patient_id} has no images."
        )

    unique_labels = np.sort(
        np.unique(labels)
    )

    label_counts = (
        pd.Series(labels)
        .value_counts()
        .reindex(
            [
                0,
                1,
                2,
                3,
            ],
            fill_value=0,
        )
        .sort_index()
    )

    modal_classes = np.sort(
        pd.Series(labels)
        .mode()
        .to_numpy(
            dtype=int
        )
    )

    modal_low = int(
        modal_classes[0]
    )

    modal_high = int(
        modal_classes[-1]
    )

    modal_tie = bool(
        len(modal_classes) > 1
    )

    median_value = float(
        np.median(labels)
    )

    median_low = int(
        np.floor(median_value)
    )

    median_high = int(
        np.ceil(median_value)
    )

    minimum_label = int(
        labels.min()
    )

    maximum_label = int(
        labels.max()
    )

    majority_count = int(
        label_counts.max()
    )

    majority_fraction = float(
        majority_count /
        len(labels)
    )

    return {
        "patient_id": int(patient_id),

        "n_images": int(
            len(labels)
        ),

        "n_unique_labels": int(
            len(unique_labels)
        ),

        "unique_labels": "|".join(
            map(
                str,
                unique_labels.tolist(),
            )
        ),

        "n_mayo_0_images": int(
            label_counts.loc[0]
        ),

        "n_mayo_1_images": int(
            label_counts.loc[1]
        ),

        "n_mayo_2_images": int(
            label_counts.loc[2]
        ),

        "n_mayo_3_images": int(
            label_counts.loc[3]
        ),

        # Primary and alternative patient-reference rules
        "modal_low": modal_low,
        "modal_high": modal_high,
        "modal_tie": modal_tie,

        "modal_classes": "|".join(
            map(
                str,
                modal_classes.tolist(),
            )
        ),

        "median_value": median_value,
        "median_low": median_low,
        "median_high": median_high,

        "minimum_label": minimum_label,
        "maximum_label": maximum_label,

        # Heterogeneity descriptors
        "label_range": int(
            maximum_label -
            minimum_label
        ),

        "majority_count": majority_count,
        "majority_fraction": majority_fraction,

        "label_entropy": (
            shannon_entropy_base2(
                labels
            )
        ),
    }


# ============================================================
# 11. Build the patient-reference audit
# ============================================================

patient_rows = []

for patient_id, patient_group in (
    trainval_df.groupby(
        "patient_id",
        sort=True,
    )
):
    patient_rows.append(
        summarize_one_patient(
            patient_id=int(patient_id),
            patient_group=patient_group,
        )
    )

patient_reference_audit = (
    pd.DataFrame(
        patient_rows
    )
    .sort_values(
        "patient_id"
    )
    .reset_index(
        drop=True
    )
)

expected_patient_count = (
    trainval_df["patient_id"]
    .nunique()
)

if len(patient_reference_audit) != expected_patient_count:
    raise RuntimeError(
        "Patient-reference audit row count does not match "
        "the number of unique development patients."
    )

if patient_reference_audit["patient_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient IDs were found in the audit."
    )


# ============================================================
# 12. Add reference-rule disagreement indicators
# ============================================================

reference_columns = [
    "modal_low",
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]

pairwise_comparisons = [
    ("modal_low", "modal_high"),
    ("modal_low", "median_low"),
    ("modal_low", "median_high"),
    ("modal_low", "maximum_label"),
    ("modal_high", "median_low"),
    ("modal_high", "median_high"),
    ("modal_high", "maximum_label"),
    ("median_low", "median_high"),
    ("median_low", "maximum_label"),
    ("median_high", "maximum_label"),
]

for rule_a, rule_b in pairwise_comparisons:
    indicator_name = (
        f"{rule_a}_vs_{rule_b}_disagree"
    )

    patient_reference_audit[
        indicator_name
    ] = (
        patient_reference_audit[rule_a]
        !=
        patient_reference_audit[rule_b]
    )


# ============================================================
# 13. Heterogeneity summary
# ============================================================

n_patients = len(
    patient_reference_audit
)

heterogeneity_summary = pd.DataFrame({
    "measure": [
        "Total patients",
        "Patients with >1 image-level MES label",
        "Patients with modal ties",
        "Patients with label range >= 1",
        "Patients with label range >= 2",
        "Patients with label range >= 3",
        "Patients with majority fraction < 0.50",
        "Patients with majority fraction < 0.67",
        "Patients with label entropy > 1.0 bit",
    ],

    "count": [
        n_patients,

        int(
            (
                patient_reference_audit[
                    "n_unique_labels"
                ] > 1
            ).sum()
        ),

        int(
            patient_reference_audit[
                "modal_tie"
            ].sum()
        ),

        int(
            (
                patient_reference_audit[
                    "label_range"
                ] >= 1
            ).sum()
        ),

        int(
            (
                patient_reference_audit[
                    "label_range"
                ] >= 2
            ).sum()
        ),

        int(
            (
                patient_reference_audit[
                    "label_range"
                ] >= 3
            ).sum()
        ),

        int(
            (
                patient_reference_audit[
                    "majority_fraction"
                ] < 0.50
            ).sum()
        ),

        int(
            (
                patient_reference_audit[
                    "majority_fraction"
                ] < 0.67
            ).sum()
        ),

        int(
            (
                patient_reference_audit[
                    "label_entropy"
                ] > 1.0
            ).sum()
        ),
    ],
})

heterogeneity_summary[
    "percent"
] = (
    100.0 *
    heterogeneity_summary["count"] /
    n_patients
)


# ============================================================
# 14. Pairwise disagreement summary
# ============================================================

disagreement_rows = []

for rule_a, rule_b in pairwise_comparisons:
    disagree = (
        patient_reference_audit[rule_a]
        !=
        patient_reference_audit[rule_b]
    )

    n_disagree = int(
        disagree.sum()
    )

    disagreement_rows.append({
        "reference_rule_a": rule_a,
        "reference_rule_b": rule_b,
        "n_disagree": n_disagree,
        "percent_disagree": float(
            100.0 *
            n_disagree /
            n_patients
        ),
    })

reference_disagreement_summary = pd.DataFrame(
    disagreement_rows
)


# ============================================================
# 15. Patient-label distributions by reference rule
# ============================================================

distribution_rows = []

for reference_rule in reference_columns:
    counts = (
        patient_reference_audit[
            reference_rule
        ]
        .value_counts()
        .reindex(
            [
                0,
                1,
                2,
                3,
            ],
            fill_value=0,
        )
        .sort_index()
    )

    for mayo_class in [
        0,
        1,
        2,
        3,
    ]:
        n_class_patients = int(
            counts.loc[
                mayo_class
            ]
        )

        distribution_rows.append({
            "reference_rule": reference_rule,
            "mayo_class": mayo_class,
            "n_patients": n_class_patients,
            "percent_patients": float(
                100.0 *
                n_class_patients /
                n_patients
            ),
        })

reference_distribution_long = pd.DataFrame(
    distribution_rows
)

reference_distribution_wide = (
    reference_distribution_long
    .pivot(
        index="reference_rule",
        columns="mayo_class",
        values="n_patients",
    )
    .rename(
        columns={
            0: "mayo_0",
            1: "mayo_1",
            2: "mayo_2",
            3: "mayo_3",
        }
    )
    .reset_index()
)


# ============================================================
# 16. Descriptive summaries
# ============================================================

majority_fraction_summary = (
    patient_reference_audit[
        "majority_fraction"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame(
        name="majority_fraction"
    )
)

label_entropy_summary = (
    patient_reference_audit[
        "label_entropy"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame(
        name="label_entropy"
    )
)

images_per_patient_summary = (
    patient_reference_audit[
        "n_images"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame(
        name="n_images"
    )
)


# ============================================================
# 17. Construct frozen patient-stratified folds
# ============================================================

PRIMARY_REFERENCE_RULE = "modal_low"

patient_fold_assignment = (
    patient_reference_audit
    .copy()
)

patient_fold_assignment[
    "patient_label"
] = (
    patient_fold_assignment[
        PRIMARY_REFERENCE_RULE
    ]
    .astype(int)
)

patient_fold_assignment[
    "cv_fold"
] = -1

stratified_folds = StratifiedKFold(
    n_splits=BASE_CONFIG[
        "num_folds"
    ],
    shuffle=True,
    random_state=BASE_CONFIG[
        "fold_seed"
    ],
)

for fold_index, (_, validation_indices) in enumerate(
    stratified_folds.split(
        np.zeros(
            len(
                patient_fold_assignment
            )
        ),
        patient_fold_assignment[
            "patient_label"
        ].to_numpy(),
    )
):
    patient_fold_assignment.loc[
        validation_indices,
        "cv_fold",
    ] = int(
        fold_index
    )

if (
    patient_fold_assignment[
        "cv_fold"
    ] < 0
).any():
    raise RuntimeError(
        "At least one patient was not assigned to a fold."
    )

patient_fold_assignment[
    "cv_fold"
] = (
    patient_fold_assignment[
        "cv_fold"
    ]
    .astype(int)
)

expected_folds = set(
    range(
        BASE_CONFIG[
            "num_folds"
        ]
    )
)

observed_folds = set(
    patient_fold_assignment[
        "cv_fold"
    ].unique()
)

if observed_folds != expected_folds:
    raise RuntimeError(
        f"Expected folds {sorted(expected_folds)}, "
        f"but found {sorted(observed_folds)}."
    )


# ============================================================
# 18. Define output paths
# ============================================================

PATIENT_REFERENCE_AUDIT_PATH = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_sensitivity.csv"
)

FOLD_ASSIGNMENT_PATH = (
    BASE_SAVE_ROOT /
    "patient_fold_assignment.csv"
)

HETEROGENEITY_SUMMARY_PATH = (
    BASE_SAVE_ROOT /
    "patient_label_heterogeneity_summary.csv"
)

DISAGREEMENT_SUMMARY_PATH = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_disagreement_summary.csv"
)

DISTRIBUTION_LONG_PATH = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_distribution_long.csv"
)

DISTRIBUTION_WIDE_PATH = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_distribution_wide.csv"
)

MAJORITY_FRACTION_SUMMARY_PATH = (
    BASE_SAVE_ROOT /
    "majority_fraction_summary.csv"
)

LABEL_ENTROPY_SUMMARY_PATH = (
    BASE_SAVE_ROOT /
    "label_entropy_summary.csv"
)

IMAGES_PER_PATIENT_SUMMARY_PATH = (
    BASE_SAVE_ROOT /
    "images_per_patient_summary.csv"
)

REFERENCE_AUDIT_JSON_PATH = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_audit_summary.json"
)

DEVELOPMENT_MANIFEST_PATH = (
    BASE_SAVE_ROOT /
    "development_image_manifest_with_folds.csv"
)

ANALYSIS_PLAN_PATH = (
    BASE_SAVE_ROOT /
    "analysis_plan_frozen_before_training.json"
)


# ============================================================
# 19. Protect frozen files when resuming after Cell 3
# ============================================================

def validate_existing_frozen_outputs(
    current_audit: pd.DataFrame,
    current_fold_assignment: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    saved_audit = pd.read_csv(
        PATIENT_REFERENCE_AUDIT_PATH
    )

    saved_folds = pd.read_csv(
        FOLD_ASSIGNMENT_PATH
    )

    audit_key_columns = [
        "patient_id",
        "modal_low",
        "modal_high",
        "median_low",
        "median_high",
        "maximum_label",
        "n_images",
        "n_unique_labels",
        "label_range",
        "modal_tie",
    ]

    fold_key_columns = [
        "patient_id",
        "patient_label",
        "cv_fold",
    ]

    saved_audit_check = (
        saved_audit[
            audit_key_columns
        ]
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    current_audit_check = (
        current_audit[
            audit_key_columns
        ]
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    saved_fold_check = (
        saved_folds[
            fold_key_columns
        ]
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    current_fold_check = (
        current_fold_assignment[
            fold_key_columns
        ]
        .sort_values(
            "patient_id"
        )
        .reset_index(
            drop=True
        )
    )

    try:
        pd.testing.assert_frame_equal(
            saved_audit_check,
            current_audit_check,
            check_dtype=False,
        )

        pd.testing.assert_frame_equal(
            saved_fold_check,
            current_fold_check,
            check_dtype=False,
        )

    except AssertionError as error:
        raise RuntimeError(
            "The current dataset-derived patient references or folds "
            "do not match the files frozen before training.\n"
            "Do not continue in the same output directory.\n"
            f"Details:\n{error}"
        )

    print(
        "Existing frozen reference and fold files were validated."
    )

    return (
        saved_audit,
        saved_folds,
    )


if ANALYSIS_PLAN_PATH.exists():
    if not PATIENT_REFERENCE_AUDIT_PATH.exists():
        raise FileNotFoundError(
            "A frozen analysis plan exists, but the saved patient "
            "reference audit is missing."
        )

    if not FOLD_ASSIGNMENT_PATH.exists():
        raise FileNotFoundError(
            "A frozen analysis plan exists, but the saved fold "
            "assignment is missing."
        )

    (
        patient_reference_audit,
        patient_fold_assignment,
    ) = validate_existing_frozen_outputs(
        current_audit=patient_reference_audit,
        current_fold_assignment=patient_fold_assignment,
    )

    patient_reference_audit[
        "patient_id"
    ] = (
        patient_reference_audit[
            "patient_id"
        ].astype(int)
    )

    patient_fold_assignment[
        "patient_id"
    ] = (
        patient_fold_assignment[
            "patient_id"
        ].astype(int)
    )

    patient_fold_assignment[
        "patient_label"
    ] = (
        patient_fold_assignment[
            "patient_label"
        ].astype(int)
    )

    patient_fold_assignment[
        "cv_fold"
    ] = (
        patient_fold_assignment[
            "cv_fold"
        ].astype(int)
    )

    print(
        "Frozen analysis plan detected. "
        "Reference and fold files were not overwritten."
    )

else:
    # Save all audit and fold files before training.
    patient_reference_audit.to_csv(
        PATIENT_REFERENCE_AUDIT_PATH,
        index=False,
    )

    patient_fold_assignment.to_csv(
        FOLD_ASSIGNMENT_PATH,
        index=False,
    )

    heterogeneity_summary.to_csv(
        HETEROGENEITY_SUMMARY_PATH,
        index=False,
    )

    reference_disagreement_summary.to_csv(
        DISAGREEMENT_SUMMARY_PATH,
        index=False,
    )

    reference_distribution_long.to_csv(
        DISTRIBUTION_LONG_PATH,
        index=False,
    )

    reference_distribution_wide.to_csv(
        DISTRIBUTION_WIDE_PATH,
        index=False,
    )

    majority_fraction_summary.to_csv(
        MAJORITY_FRACTION_SUMMARY_PATH
    )

    label_entropy_summary.to_csv(
        LABEL_ENTROPY_SUMMARY_PATH
    )

    images_per_patient_summary.to_csv(
        IMAGES_PER_PATIENT_SUMMARY_PATH
    )

    print(
        "Saved patient-reference audit and frozen fold files."
    )


# ============================================================
# 20. Save pairwise cross-tabulations
# ============================================================

for alternative_rule in [
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]:
    cross_tabulation = pd.crosstab(
        patient_reference_audit[
            "modal_low"
        ],
        patient_reference_audit[
            alternative_rule
        ],
        rownames=[
            "modal_low"
        ],
        colnames=[
            alternative_rule
        ],
        dropna=False,
    ).reindex(
        index=[
            0,
            1,
            2,
            3,
        ],
        columns=[
            0,
            1,
            2,
            3,
        ],
        fill_value=0,
    )

    cross_tabulation.to_csv(
        BASE_SAVE_ROOT /
        f"crosstab_modal_low_vs_{alternative_rule}.csv"
    )


# ============================================================
# 21. Apply the frozen folds and patient reference to images
# ============================================================

fold_map = (
    patient_fold_assignment
    .set_index(
        "patient_id"
    )[
        "cv_fold"
    ]
    .to_dict()
)

patient_label_map = (
    patient_fold_assignment
    .set_index(
        "patient_id"
    )[
        "patient_label"
    ]
    .to_dict()
)

trainval_df[
    "cv_fold"
] = (
    trainval_df[
        "patient_id"
    ]
    .map(
        fold_map
    )
)

trainval_df[
    "patient_y_true"
] = (
    trainval_df[
        "patient_id"
    ]
    .map(
        patient_label_map
    )
)

if trainval_df["cv_fold"].isna().any():
    raise RuntimeError(
        "Some development images could not be assigned to a fold."
    )

if trainval_df["patient_y_true"].isna().any():
    raise RuntimeError(
        "Some development images lack a derived patient reference."
    )

trainval_df[
    "cv_fold"
] = (
    trainval_df[
        "cv_fold"
    ]
    .astype(int)
)

trainval_df[
    "patient_y_true"
] = (
    trainval_df[
        "patient_y_true"
    ]
    .astype(int)
)

if (
    trainval_df
    .groupby(
        "patient_id"
    )[
        "cv_fold"
    ]
    .nunique()
    .max()
    != 1
):
    raise RuntimeError(
        "A patient appears in more than one development fold."
    )

if (
    trainval_df
    .groupby(
        "patient_id"
    )[
        "patient_y_true"
    ]
    .nunique()
    .max()
    != 1
):
    raise RuntimeError(
        "A patient has inconsistent derived reference labels."
    )

trainval_df[
    [
        "relative_path",
        "filename",
        "label",
        "patient_id",
        "image_index",
        "cv_fold",
        "patient_y_true",
    ]
].to_csv(
    DEVELOPMENT_MANIFEST_PATH,
    index=False,
)


# ============================================================
# 22. Save compact JSON audit summary
# ============================================================

compact_reference_summary = {
    "development_images": int(
        len(trainval_df)
    ),

    "development_patients": int(
        trainval_df[
            "patient_id"
        ].nunique()
    ),

    "primary_reference_rule": (
        PRIMARY_REFERENCE_RULE
    ),

    "primary_reference_definition": (
        "Modal image-level MES; lower modal class is used "
        "when multiple modal classes are tied."
    ),

    "patients_with_multiple_image_labels": int(
        (
            patient_reference_audit[
                "n_unique_labels"
            ] > 1
        ).sum()
    ),

    "patients_with_modal_ties": int(
        patient_reference_audit[
            "modal_tie"
        ].sum()
    ),

    "patients_modal_low_vs_modal_high_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ]
            !=
            patient_reference_audit[
                "modal_high"
            ]
        ).sum()
    ),

    "patients_modal_low_vs_median_low_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ]
            !=
            patient_reference_audit[
                "median_low"
            ]
        ).sum()
    ),

    "patients_modal_low_vs_median_high_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ]
            !=
            patient_reference_audit[
                "median_high"
            ]
        ).sum()
    ),

    "patients_modal_low_vs_maximum_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ]
            !=
            patient_reference_audit[
                "maximum_label"
            ]
        ).sum()
    ),

    "mean_majority_fraction": float(
        patient_reference_audit[
            "majority_fraction"
        ].mean()
    ),

    "median_majority_fraction": float(
        patient_reference_audit[
            "majority_fraction"
        ].median()
    ),

    "mean_label_entropy_bits": float(
        patient_reference_audit[
            "label_entropy"
        ].mean()
    ),

    "median_label_entropy_bits": float(
        patient_reference_audit[
            "label_entropy"
        ].median()
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

with open(
    REFERENCE_AUDIT_JSON_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        compact_reference_summary,
        file,
        indent=2,
    )


# ============================================================
# 23. Save environment metadata
# ============================================================

def safe_package_version(
    package_name: str,
):
    try:
        return version(
            package_name
        )

    except PackageNotFoundError:
        return None


environment_metadata = {
    "python": platform.python_version(),
    "platform": platform.platform(),

    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "timm": timm.__version__,

    "numpy": np.__version__,
    "pandas": pd.__version__,

    "scikit_learn": safe_package_version(
        "scikit-learn"
    ),

    "pillow": safe_package_version(
        "Pillow"
    ),

    "cuda_available": bool(
        torch.cuda.is_available()
    ),

    "cuda_version": (
        torch.version.cuda
    ),

    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,
}

environment_metadata_path = (
    BASE_SAVE_ROOT /
    "environment_metadata.json"
)

with open(
    environment_metadata_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_metadata,
        file,
        indent=2,
    )


# ============================================================
# 24. Display the complete audit
# ============================================================

print(
    "\n" +
    "=" * 80
)

print(
    "DEVELOPMENT DATA AND PATIENT-REFERENCE AUDIT"
)

print(
    "=" * 80
)

print(
    "\nDevelopment images:",
    len(trainval_df),
)

print(
    "Development patients:",
    trainval_df[
        "patient_id"
    ].nunique(),
)

print(
    "\nImage counts by MES:"
)

display(
    trainval_df[
        "label"
    ]
    .value_counts()
    .sort_index()
    .rename(
        "n_images"
    )
)

print(
    "\nPatient counts by primary derived reference:"
)

display(
    patient_reference_audit[
        PRIMARY_REFERENCE_RULE
    ]
    .value_counts()
    .sort_index()
    .rename(
        "n_patients"
    )
)

print(
    "\nPatient-label heterogeneity summary:"
)

display(
    heterogeneity_summary
)

print(
    "\nPairwise patient-reference disagreement:"
)

display(
    reference_disagreement_summary
)

print(
    "\nPatient-label distributions by reference rule:"
)

display(
    reference_distribution_wide
)

print(
    "\nMajority-fraction summary:"
)

display(
    majority_fraction_summary
)

print(
    "\nLabel-entropy summary:"
)

display(
    label_entropy_summary
)

print(
    "\nImages-per-patient summary:"
)

display(
    images_per_patient_summary
)

print(
    "\nFrozen patient counts by fold and MES:"
)

display(
    patient_fold_assignment
    .groupby(
        [
            "cv_fold",
            "patient_label",
        ]
    )
    .size()
    .reset_index(
        name="n_patients"
    )
    .sort_values(
        [
            "cv_fold",
            "patient_label",
        ]
    )
)

print(
    "\nEnvironment metadata:"
)

print(
    json.dumps(
        environment_metadata,
        indent=2,
    )
)


# ============================================================
# 25. Final status
# ============================================================

print(
    "\n" +
    "=" * 80
)

print(
    "CELL 1 COMPLETED SUCCESSFULLY"
)

print(
    "=" * 80
)

print(
    "Primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Frozen fold file:",
    FOLD_ASSIGNMENT_PATH,
)

print(
    "Patient-reference audit file:",
    PATIENT_REFERENCE_AUDIT_PATH,
)

print(
    "Development manifest:",
    DEVELOPMENT_MANIFEST_PATH,
)

print(
    "Environment metadata:",
    environment_metadata_path,
)

print(
    "Official test data loaded:",
    False,
)

print(
    "Official test data evaluated:",
    False,
)

print(
    "\nNext step: run the revised complete Cell 2."
)

In [ ]:
# ============================================================
# Additional Patient-Reference Sensitivity Audit
# ------------------------------------------------------------
# Run this cell AFTER Cell 1 of:
#   01_vit_loss_screening_development_only.ipynb
#
# Required objects already defined:
#   - trainval_df
#   - BASE_SAVE_ROOT
#
# Purpose:
#   Audit how the derived patient-level MES reference changes
#   under alternative rules:
#
#     1) modal_low
#     2) modal_high
#     3) median_low
#     4) median_high
#     5) maximum_label
#
# Important:
#   - This code does NOT modify the primary fold assignment.
#   - This code does NOT load or evaluate the official test set.
#   - The primary analysis can remain based on modal_low.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Validate required inputs
# ------------------------------------------------------------
required_columns = {"patient_id", "label"}

missing_columns = required_columns - set(trainval_df.columns)
if missing_columns:
    raise ValueError(
        f"trainval_df is missing required columns: {sorted(missing_columns)}"
    )

if len(trainval_df) == 0:
    raise ValueError("trainval_df is empty.")

if trainval_df["patient_id"].isna().any():
    raise ValueError("trainval_df contains missing patient_id values.")

if trainval_df["label"].isna().any():
    raise ValueError("trainval_df contains missing label values.")

trainval_df = trainval_df.copy()
trainval_df["patient_id"] = trainval_df["patient_id"].astype(int)
trainval_df["label"] = trainval_df["label"].astype(int)

valid_labels = {0, 1, 2, 3}
observed_labels = set(trainval_df["label"].unique())

if not observed_labels.issubset(valid_labels):
    raise ValueError(
        f"Unexpected MES labels detected: {sorted(observed_labels)}"
    )

BASE_SAVE_ROOT = Path(BASE_SAVE_ROOT)
BASE_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

print("Development images:", len(trainval_df))
print("Development patients:", trainval_df["patient_id"].nunique())


# ------------------------------------------------------------
# 2. Helper functions
# ------------------------------------------------------------
def shannon_entropy_base2(labels: np.ndarray) -> float:
    """
    Compute Shannon entropy in bits from integer class labels.
    """
    labels = np.asarray(labels, dtype=int)

    if len(labels) == 0:
        return np.nan

    counts = pd.Series(labels).value_counts().sort_index()
    probabilities = counts.to_numpy(dtype=float) / counts.sum()

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    )

    # Avoid displaying negative zero.
    if abs(entropy) < 1e-12:
        entropy = 0.0

    return float(entropy)


def summarize_one_patient(
    patient_id: int,
    patient_group: pd.DataFrame,
) -> dict:
    """
    Summarize all image-level MES labels belonging to one patient.
    """
    labels = (
        patient_group["label"]
        .astype(int)
        .to_numpy()
    )

    if len(labels) == 0:
        raise ValueError(
            f"Patient {patient_id} has no image-level labels."
        )

    unique_labels = np.sort(np.unique(labels))

    label_counts = (
        pd.Series(labels)
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
        .sort_index()
    )

    modal_classes = np.sort(
        pd.Series(labels)
        .mode()
        .to_numpy(dtype=int)
    )

    modal_low = int(modal_classes[0])
    modal_high = int(modal_classes[-1])
    modal_tie = bool(len(modal_classes) > 1)

    median_value = float(np.median(labels))
    median_low = int(np.floor(median_value))
    median_high = int(np.ceil(median_value))

    minimum_label = int(labels.min())
    maximum_label = int(labels.max())

    majority_count = int(label_counts.max())
    majority_fraction = float(
        majority_count / len(labels)
    )

    entropy = shannon_entropy_base2(labels)

    row = {
        "patient_id": int(patient_id),

        # Sample-size and heterogeneity descriptors
        "n_images": int(len(labels)),
        "n_unique_labels": int(len(unique_labels)),
        "unique_labels": "|".join(
            map(str, unique_labels.tolist())
        ),

        # Per-class image counts
        "n_mayo_0_images": int(label_counts.loc[0]),
        "n_mayo_1_images": int(label_counts.loc[1]),
        "n_mayo_2_images": int(label_counts.loc[2]),
        "n_mayo_3_images": int(label_counts.loc[3]),

        # Primary and alternative patient-reference rules
        "modal_low": modal_low,
        "modal_high": modal_high,
        "modal_tie": modal_tie,
        "modal_classes": "|".join(
            map(str, modal_classes.tolist())
        ),

        "median_value": median_value,
        "median_low": median_low,
        "median_high": median_high,

        "minimum_label": minimum_label,
        "maximum_label": maximum_label,

        # Heterogeneity summaries
        "label_range": int(
            maximum_label - minimum_label
        ),
        "majority_count": majority_count,
        "majority_fraction": majority_fraction,
        "label_entropy": entropy,
    }

    return row


# ------------------------------------------------------------
# 3. Build one-row-per-patient audit table
# ------------------------------------------------------------
patient_rows = []

for patient_id, patient_group in trainval_df.groupby(
    "patient_id",
    sort=True,
):
    patient_rows.append(
        summarize_one_patient(
            patient_id=int(patient_id),
            patient_group=patient_group,
        )
    )

patient_reference_audit = pd.DataFrame(patient_rows)

patient_reference_audit = (
    patient_reference_audit
    .sort_values("patient_id")
    .reset_index(drop=True)
)

expected_patient_count = (
    trainval_df["patient_id"].nunique()
)

if len(patient_reference_audit) != expected_patient_count:
    raise RuntimeError(
        "Patient audit row count does not match "
        "the number of unique patients."
    )

if patient_reference_audit["patient_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient IDs found in patient_reference_audit."
    )


# ------------------------------------------------------------
# 4. Add disagreement indicators
# ------------------------------------------------------------
reference_columns = [
    "modal_low",
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]

# Comparisons against the primary reference rule.
for alternative in [
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]:
    comparison_name = (
        f"modal_low_vs_{alternative}_disagree"
    )

    patient_reference_audit[comparison_name] = (
        patient_reference_audit["modal_low"] !=
        patient_reference_audit[alternative]
    )

# Additional useful comparisons.
patient_reference_audit[
    "median_low_vs_median_high_disagree"
] = (
    patient_reference_audit["median_low"] !=
    patient_reference_audit["median_high"]
)

patient_reference_audit[
    "median_low_vs_maximum_disagree"
] = (
    patient_reference_audit["median_low"] !=
    patient_reference_audit["maximum_label"]
)

patient_reference_audit[
    "modal_high_vs_maximum_disagree"
] = (
    patient_reference_audit["modal_high"] !=
    patient_reference_audit["maximum_label"]
)


# ------------------------------------------------------------
# 5. Heterogeneity summary
# ------------------------------------------------------------
n_patients = len(patient_reference_audit)

heterogeneity_summary = pd.DataFrame({
    "measure": [
        "Total patients",
        "Patients with >1 image-level MES label",
        "Patients with modal ties",
        "Patients with label range >= 1",
        "Patients with label range >= 2",
        "Patients with label range >= 3",
        "Patients with majority fraction < 0.50",
        "Patients with majority fraction < 0.67",
        "Patients with label entropy > 1.0 bit",
    ],
    "count": [
        n_patients,
        int(
            (
                patient_reference_audit[
                    "n_unique_labels"
                ] > 1
            ).sum()
        ),
        int(
            patient_reference_audit[
                "modal_tie"
            ].sum()
        ),
        int(
            (
                patient_reference_audit[
                    "label_range"
                ] >= 1
            ).sum()
        ),
        int(
            (
                patient_reference_audit[
                    "label_range"
                ] >= 2
            ).sum()
        ),
        int(
            (
                patient_reference_audit[
                    "label_range"
                ] >= 3
            ).sum()
        ),
        int(
            (
                patient_reference_audit[
                    "majority_fraction"
                ] < 0.50
            ).sum()
        ),
        int(
            (
                patient_reference_audit[
                    "majority_fraction"
                ] < 0.67
            ).sum()
        ),
        int(
            (
                patient_reference_audit[
                    "label_entropy"
                ] > 1.0
            ).sum()
        ),
    ],
})

heterogeneity_summary["percent"] = (
    100.0 *
    heterogeneity_summary["count"] /
    n_patients
)

# Do not report percentage for "Total patients".
heterogeneity_summary.loc[
    heterogeneity_summary["measure"] == "Total patients",
    "percent",
] = 100.0


# ------------------------------------------------------------
# 6. Pairwise reference-rule disagreement summary
# ------------------------------------------------------------
pairwise_comparisons = [
    ("modal_low", "modal_high"),
    ("modal_low", "median_low"),
    ("modal_low", "median_high"),
    ("modal_low", "maximum_label"),
    ("modal_high", "median_low"),
    ("modal_high", "median_high"),
    ("modal_high", "maximum_label"),
    ("median_low", "median_high"),
    ("median_low", "maximum_label"),
    ("median_high", "maximum_label"),
]

disagreement_rows = []

for rule_a, rule_b in pairwise_comparisons:
    disagree = (
        patient_reference_audit[rule_a] !=
        patient_reference_audit[rule_b]
    )

    n_disagree = int(disagree.sum())

    disagreement_rows.append({
        "reference_rule_a": rule_a,
        "reference_rule_b": rule_b,
        "n_disagree": n_disagree,
        "percent_disagree": float(
            100.0 * n_disagree / n_patients
        ),
    })

reference_disagreement_summary = pd.DataFrame(
    disagreement_rows
)


# ------------------------------------------------------------
# 7. Patient-label distribution by reference rule
# ------------------------------------------------------------
distribution_rows = []

for reference_rule in reference_columns:
    counts = (
        patient_reference_audit[reference_rule]
        .value_counts()
        .reindex([0, 1, 2, 3], fill_value=0)
        .sort_index()
    )

    for mayo_class in [0, 1, 2, 3]:
        n_class_patients = int(
            counts.loc[mayo_class]
        )

        distribution_rows.append({
            "reference_rule": reference_rule,
            "mayo_class": mayo_class,
            "n_patients": n_class_patients,
            "percent_patients": float(
                100.0 *
                n_class_patients /
                n_patients
            ),
        })

reference_distribution_long = pd.DataFrame(
    distribution_rows
)

reference_distribution_wide = (
    reference_distribution_long
    .pivot(
        index="reference_rule",
        columns="mayo_class",
        values="n_patients",
    )
    .rename(
        columns={
            0: "mayo_0",
            1: "mayo_1",
            2: "mayo_2",
            3: "mayo_3",
        }
    )
    .reset_index()
)


# ------------------------------------------------------------
# 8. Cross-tabulations against modal-low
# ------------------------------------------------------------
modal_low_crosstabs = {}

for alternative_rule in [
    "modal_high",
    "median_low",
    "median_high",
    "maximum_label",
]:
    crosstab = pd.crosstab(
        patient_reference_audit["modal_low"],
        patient_reference_audit[alternative_rule],
        rownames=["modal_low"],
        colnames=[alternative_rule],
        dropna=False,
    ).reindex(
        index=[0, 1, 2, 3],
        columns=[0, 1, 2, 3],
        fill_value=0,
    )

    modal_low_crosstabs[alternative_rule] = crosstab


# ------------------------------------------------------------
# 9. Descriptive summaries
# ------------------------------------------------------------
majority_fraction_summary = (
    patient_reference_audit[
        "majority_fraction"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame(name="majority_fraction")
)

label_entropy_summary = (
    patient_reference_audit[
        "label_entropy"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame(name="label_entropy")
)

images_per_patient_summary = (
    patient_reference_audit[
        "n_images"
    ]
    .describe(
        percentiles=[
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
        ]
    )
    .to_frame(name="n_images")
)


# ------------------------------------------------------------
# 10. Print audit results
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PATIENT REFERENCE RULE AUDIT")
print("=" * 70)

print("\nHeterogeneity summary:")
display(heterogeneity_summary)

print("\nPairwise reference-rule disagreement:")
display(reference_disagreement_summary)

print("\nPatient-label distributions by reference rule:")
display(reference_distribution_wide)

for reference_rule in reference_columns:
    print(f"\nDistribution: {reference_rule}")

    display(
        reference_distribution_long[
            reference_distribution_long[
                "reference_rule"
            ] == reference_rule
        ][
            [
                "mayo_class",
                "n_patients",
                "percent_patients",
            ]
        ].reset_index(drop=True)
    )

print("\nModal-low cross-tabulations:")

for alternative_rule, crosstab in modal_low_crosstabs.items():
    print(f"\nmodal_low vs {alternative_rule}")
    display(crosstab)

print("\nMajority-fraction summary:")
display(majority_fraction_summary)

print("\nLabel-entropy summary:")
display(label_entropy_summary)

print("\nImages-per-patient summary:")
display(images_per_patient_summary)


# ------------------------------------------------------------
# 11. Save all audit outputs
# ------------------------------------------------------------
patient_reference_audit_path = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_sensitivity.csv"
)

heterogeneity_summary_path = (
    BASE_SAVE_ROOT /
    "patient_label_heterogeneity_summary.csv"
)

disagreement_summary_path = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_disagreement_summary.csv"
)

distribution_long_path = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_distribution_long.csv"
)

distribution_wide_path = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_distribution_wide.csv"
)

majority_fraction_summary_path = (
    BASE_SAVE_ROOT /
    "majority_fraction_summary.csv"
)

label_entropy_summary_path = (
    BASE_SAVE_ROOT /
    "label_entropy_summary.csv"
)

images_per_patient_summary_path = (
    BASE_SAVE_ROOT /
    "images_per_patient_summary.csv"
)

patient_reference_audit.to_csv(
    patient_reference_audit_path,
    index=False,
)

heterogeneity_summary.to_csv(
    heterogeneity_summary_path,
    index=False,
)

reference_disagreement_summary.to_csv(
    disagreement_summary_path,
    index=False,
)

reference_distribution_long.to_csv(
    distribution_long_path,
    index=False,
)

reference_distribution_wide.to_csv(
    distribution_wide_path,
    index=False,
)

majority_fraction_summary.to_csv(
    majority_fraction_summary_path
)

label_entropy_summary.to_csv(
    label_entropy_summary_path
)

images_per_patient_summary.to_csv(
    images_per_patient_summary_path
)

for alternative_rule, crosstab in modal_low_crosstabs.items():
    crosstab_path = (
        BASE_SAVE_ROOT /
        f"crosstab_modal_low_vs_{alternative_rule}.csv"
    )

    crosstab.to_csv(crosstab_path)


# ------------------------------------------------------------
# 12. Save compact JSON summary
# ------------------------------------------------------------
compact_summary = {
    "n_images": int(len(trainval_df)),
    "n_patients": int(n_patients),

    "primary_reference_rule": (
        "modal_low"
    ),

    "primary_reference_definition": (
        "Modal image-level MES; lower modal class "
        "used when multiple modal classes were tied."
    ),

    "n_patients_with_multiple_image_labels": int(
        (
            patient_reference_audit[
                "n_unique_labels"
            ] > 1
        ).sum()
    ),

    "n_patients_with_modal_ties": int(
        patient_reference_audit[
            "modal_tie"
        ].sum()
    ),

    "n_modal_low_vs_modal_high_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ] !=
            patient_reference_audit[
                "modal_high"
            ]
        ).sum()
    ),

    "n_modal_low_vs_median_low_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ] !=
            patient_reference_audit[
                "median_low"
            ]
        ).sum()
    ),

    "n_modal_low_vs_median_high_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ] !=
            patient_reference_audit[
                "median_high"
            ]
        ).sum()
    ),

    "n_modal_low_vs_maximum_disagree": int(
        (
            patient_reference_audit[
                "modal_low"
            ] !=
            patient_reference_audit[
                "maximum_label"
            ]
        ).sum()
    ),

    "mean_majority_fraction": float(
        patient_reference_audit[
            "majority_fraction"
        ].mean()
    ),

    "median_majority_fraction": float(
        patient_reference_audit[
            "majority_fraction"
        ].median()
    ),

    "mean_label_entropy_bits": float(
        patient_reference_audit[
            "label_entropy"
        ].mean()
    ),

    "median_label_entropy_bits": float(
        patient_reference_audit[
            "label_entropy"
        ].median()
    ),

    "output_files": {
        "patient_reference_audit": str(
            patient_reference_audit_path
        ),
        "heterogeneity_summary": str(
            heterogeneity_summary_path
        ),
        "disagreement_summary": str(
            disagreement_summary_path
        ),
        "distribution_long": str(
            distribution_long_path
        ),
        "distribution_wide": str(
            distribution_wide_path
        ),
    },
}

compact_summary_path = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_audit_summary.json"
)

with open(
    compact_summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        compact_summary,
        file,
        indent=2,
    )


# ------------------------------------------------------------
# 13. Final output report
# ------------------------------------------------------------
print("\n" + "=" * 70)
print("PATIENT REFERENCE AUDIT FILES SAVED")
print("=" * 70)

saved_paths = [
    patient_reference_audit_path,
    heterogeneity_summary_path,
    disagreement_summary_path,
    distribution_long_path,
    distribution_wide_path,
    majority_fraction_summary_path,
    label_entropy_summary_path,
    images_per_patient_summary_path,
    compact_summary_path,
]

for saved_path in saved_paths:
    print(saved_path)

print(
    "\nPrimary fold assignment was not modified. "
    "Continue to use modal_low as the primary derived "
    "patient-level reference unless the analysis plan "
    "is formally changed and documented."
)

In [ ]:
# ============================================================
# CELL 2
# LIMUC ViT Loss Screening — Core Utilities
# ------------------------------------------------------------
# Requirements from Cell 1:
#   trainval_df
#   BASE_CONFIG
#   EXPERIMENT_PRESETS
#   EXPERIMENTS_TO_RUN
#   TRAIN_SEEDS
#   BASE_SAVE_ROOT
#
# Requirements from the patient-reference audit:
#   patient_fold_assignment.csv
#   patient_reference_rule_sensitivity.csv
#
# This cell:
#   1) verifies the frozen modal-low patient reference
#   2) reloads and applies the same patient folds
#   3) defines datasets/transforms/losses
#   4) defines image- and patient-level evaluation
#   5) defines patient-level checkpoint selection
#   6) defines stratified patient bootstrap intervals
#
# The official LIMUC test set is never loaded here.
# ============================================================

import os
import json
import math
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm

from sklearn.metrics import (
    cohen_kappa_score,
    f1_score,
    accuracy_score,
    mean_absolute_error,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# ============================================================
# 1. Validate objects inherited from Cell 1
# ============================================================

REQUIRED_GLOBAL_OBJECTS = [
    "trainval_df",
    "BASE_CONFIG",
    "EXPERIMENT_PRESETS",
    "EXPERIMENTS_TO_RUN",
    "TRAIN_SEEDS",
    "BASE_SAVE_ROOT",
]

missing_objects = [
    name
    for name in REQUIRED_GLOBAL_OBJECTS
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Cell 1 must be run before Cell 2. "
        f"Missing objects: {missing_objects}"
    )

BASE_SAVE_ROOT = Path(BASE_SAVE_ROOT)
BASE_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

PRIMARY_REFERENCE_RULE = "modal_low"

FOLD_ASSIGNMENT_PATH = (
    BASE_SAVE_ROOT /
    "patient_fold_assignment.csv"
)

REFERENCE_AUDIT_PATH = (
    BASE_SAVE_ROOT /
    "patient_reference_rule_sensitivity.csv"
)

if not FOLD_ASSIGNMENT_PATH.exists():
    raise FileNotFoundError(
        "Frozen fold assignment was not found:\n"
        f"{FOLD_ASSIGNMENT_PATH}"
    )

if not REFERENCE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        "Patient-reference audit was not found:\n"
        f"{REFERENCE_AUDIT_PATH}"
    )


# ============================================================
# 2. Reproducibility helpers
# ============================================================

def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def seed_worker(worker_id: int) -> None:
    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(int(BASE_CONFIG["seed"]))

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass


# ============================================================
# 3. Reload and verify the frozen patient reference and folds
# ============================================================

fold_assignment_check = pd.read_csv(
    FOLD_ASSIGNMENT_PATH
)

reference_audit_check = pd.read_csv(
    REFERENCE_AUDIT_PATH
)

required_fold_columns = {
    "patient_id",
    "patient_label",
    "cv_fold",
}

required_audit_columns = {
    "patient_id",
    PRIMARY_REFERENCE_RULE,
}

missing_fold_columns = (
    required_fold_columns -
    set(fold_assignment_check.columns)
)

missing_audit_columns = (
    required_audit_columns -
    set(reference_audit_check.columns)
)

if missing_fold_columns:
    raise ValueError(
        "Fold assignment file is missing columns: "
        f"{sorted(missing_fold_columns)}"
    )

if missing_audit_columns:
    raise ValueError(
        "Reference audit file is missing columns: "
        f"{sorted(missing_audit_columns)}"
    )

fold_assignment_check["patient_id"] = (
    fold_assignment_check["patient_id"]
    .astype(int)
)

fold_assignment_check["patient_label"] = (
    fold_assignment_check["patient_label"]
    .astype(int)
)

fold_assignment_check["cv_fold"] = (
    fold_assignment_check["cv_fold"]
    .astype(int)
)

reference_audit_check["patient_id"] = (
    reference_audit_check["patient_id"]
    .astype(int)
)

reference_audit_check[PRIMARY_REFERENCE_RULE] = (
    reference_audit_check[PRIMARY_REFERENCE_RULE]
    .astype(int)
)

reference_check = (
    fold_assignment_check[
        [
            "patient_id",
            "patient_label",
        ]
    ]
    .merge(
        reference_audit_check[
            [
                "patient_id",
                PRIMARY_REFERENCE_RULE,
            ]
        ],
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )
)

expected_patient_count = (
    trainval_df["patient_id"]
    .nunique()
)

if len(reference_check) != expected_patient_count:
    raise RuntimeError(
        "Fold assignment/reference audit patient count "
        "does not match the development dataset.\n"
        f"Expected: {expected_patient_count}\n"
        f"Found: {len(reference_check)}"
    )

reference_matches = (
    reference_check["patient_label"]
    ==
    reference_check[PRIMARY_REFERENCE_RULE]
)

if not reference_matches.all():
    mismatch_df = (
        reference_check[
            ~reference_matches
        ]
        .head(20)
    )

    raise RuntimeError(
        "Frozen fold assignment does not match "
        "the modal-low patient reference.\n"
        f"Examples:\n{mismatch_df}"
    )

if fold_assignment_check["patient_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient IDs were found in "
        "patient_fold_assignment.csv"
    )

if reference_audit_check["patient_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate patient IDs were found in "
        "patient_reference_rule_sensitivity.csv"
    )

valid_folds = set(
    range(int(BASE_CONFIG["num_folds"]))
)

observed_folds = set(
    fold_assignment_check["cv_fold"].unique()
)

if observed_folds != valid_folds:
    raise RuntimeError(
        f"Expected folds {sorted(valid_folds)}, "
        f"but found {sorted(observed_folds)}"
    )

print(
    "Verified primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Verified development patients:",
    len(reference_check),
)

print(
    "Verified folds:",
    sorted(observed_folds),
)


# ============================================================
# 4. Apply the frozen folds and patient references
# ============================================================

trainval_df = trainval_df.copy()

trainval_df["patient_id"] = (
    trainval_df["patient_id"]
    .astype(int)
)

trainval_df["label"] = (
    trainval_df["label"]
    .astype(int)
)

fold_map = (
    fold_assignment_check
    .set_index("patient_id")["cv_fold"]
    .to_dict()
)

patient_label_map = (
    fold_assignment_check
    .set_index("patient_id")["patient_label"]
    .to_dict()
)

trainval_df["cv_fold"] = (
    trainval_df["patient_id"]
    .map(fold_map)
)

trainval_df["patient_y_true"] = (
    trainval_df["patient_id"]
    .map(patient_label_map)
)

if trainval_df["cv_fold"].isna().any():
    bad_patients = (
        trainval_df.loc[
            trainval_df["cv_fold"].isna(),
            "patient_id",
        ]
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Some development patients are absent from "
        "the frozen fold file:\n"
        f"{bad_patients[:20]}"
    )

if trainval_df["patient_y_true"].isna().any():
    bad_patients = (
        trainval_df.loc[
            trainval_df["patient_y_true"].isna(),
            "patient_id",
        ]
        .unique()
        .tolist()
    )

    raise RuntimeError(
        "Some development patients do not have "
        "a frozen patient-level reference:\n"
        f"{bad_patients[:20]}"
    )

trainval_df["cv_fold"] = (
    trainval_df["cv_fold"]
    .astype(int)
)

trainval_df["patient_y_true"] = (
    trainval_df["patient_y_true"]
    .astype(int)
)

if (
    trainval_df
    .groupby("patient_id")["cv_fold"]
    .nunique()
    .max()
    != 1
):
    raise RuntimeError(
        "A patient appears in more than one fold."
    )

if (
    trainval_df
    .groupby("patient_id")["patient_y_true"]
    .nunique()
    .max()
    != 1
):
    raise RuntimeError(
        "A patient has inconsistent derived reference labels."
    )

print("\nFrozen patient counts by fold and MES:")

display(
    fold_assignment_check
    .groupby(
        [
            "cv_fold",
            "patient_label",
        ]
    )
    .size()
    .reset_index(name="n_patients")
    .sort_values(
        [
            "cv_fold",
            "patient_label",
        ]
    )
)


# ============================================================
# 5. Dataset
# ============================================================

class LIMUCDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform=None,
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["path"]

        with Image.open(image_path) as image:
            image = image.convert("RGB")

        if self.transform is not None:
            image = self.transform(image)

        return {
            "image": image,

            # Image-level training target
            "label": torch.tensor(
                int(row["label"]),
                dtype=torch.long,
            ),

            "patient_id": torch.tensor(
                int(row["patient_id"]),
                dtype=torch.long,
            ),

            # Derived patient-level reference
            "patient_y_true": torch.tensor(
                int(row["patient_y_true"]),
                dtype=torch.long,
            ),

            "path": image_path,
        }


# ============================================================
# 6. Transforms
# ============================================================

def build_train_transform(
    image_size: int,
):
    return transforms.Compose([
        transforms.RandomResizedCrop(
            size=(
                image_size,
                image_size,
            ),
            scale=(
                0.75,
                1.0,
            ),
            ratio=(
                0.90,
                1.10,
            ),
        ),

        transforms.RandomHorizontalFlip(
            p=0.5
        ),

        transforms.RandomRotation(
            degrees=8
        ),

        transforms.ColorJitter(
            brightness=0.12,
            contrast=0.12,
            saturation=0.08,
            hue=0.02,
        ),

        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(
                0.1,
                1.2,
            ),
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


def build_eval_transform(
    image_size: int,
):
    return transforms.Compose([
        transforms.Resize(
            (
                image_size,
                image_size,
            )
        ),

        transforms.ToTensor(),

        transforms.Normalize(
            mean=[
                0.485,
                0.456,
                0.406,
            ],
            std=[
                0.229,
                0.224,
                0.225,
            ],
        ),
    ])


# ============================================================
# 7. Metrics
# ============================================================

def compute_metrics(
    y_true,
    y_pred,
    num_classes: int = 4,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_pred = np.asarray(
        y_pred,
        dtype=int,
    )

    if len(y_true) != len(y_pred):
        raise ValueError(
            "y_true and y_pred have different lengths."
        )

    if len(y_true) == 0:
        raise ValueError(
            "Cannot compute metrics on an empty set."
        )

    labels = list(
        range(num_classes)
    )

    qwk = cohen_kappa_score(
        y_true,
        y_pred,
        labels=labels,
        weights="quadratic",
    )

    if not np.isfinite(qwk):
        qwk = np.nan

    return {
        "qwk": float(qwk),
        "mae": float(
            mean_absolute_error(
                y_true,
                y_pred,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=labels,
                average="macro",
                zero_division=0,
            )
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
    }


def make_confusion_df(
    y_true,
    y_pred,
    num_classes: int = 4,
):
    labels = list(
        range(num_classes)
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=labels,
    )

    return pd.DataFrame(
        matrix,
        index=[
            f"true_{label}"
            for label in labels
        ],
        columns=[
            f"pred_{label}"
            for label in labels
        ],
    )


# ============================================================
# 8. Mean-probability patient aggregation
# ============================================================

def aggregate_mean_probabilities(
    pred_df: pd.DataFrame,
    num_classes: int = 4,
):
    prob_cols = [
        f"prob_{class_index}"
        for class_index in range(
            num_classes
        )
    ]

    required_columns = {
        "patient_id",
        "patient_y_true",
        "path",
        *prob_cols,
    }

    missing_columns = (
        required_columns -
        set(pred_df.columns)
    )

    if missing_columns:
        raise ValueError(
            "Prediction dataframe is missing columns: "
            f"{sorted(missing_columns)}"
        )

    patient_reference_counts = (
        pred_df
        .groupby("patient_id")[
            "patient_y_true"
        ]
        .nunique()
    )

    inconsistent_patients = (
        patient_reference_counts[
            patient_reference_counts > 1
        ]
        .index
        .tolist()
    )

    if inconsistent_patients:
        raise ValueError(
            "Inconsistent patient-level references for patients: "
            f"{inconsistent_patients[:20]}"
        )

    probability_df = (
        pred_df
        .groupby(
            "patient_id",
            as_index=False,
        )[prob_cols]
        .mean()
    )

    metadata_df = (
        pred_df
        .groupby(
            "patient_id",
            as_index=False,
        )
        .agg(
            y_true=(
                "patient_y_true",
                "first",
            ),
            n_images=(
                "path",
                "size",
            ),
        )
    )

    patient_df = metadata_df.merge(
        probability_df,
        on="patient_id",
        how="inner",
        validate="one_to_one",
    )

    patient_df["y_true"] = (
        patient_df["y_true"]
        .astype(int)
    )

    patient_df["n_images"] = (
        patient_df["n_images"]
        .astype(int)
    )

    patient_df["y_pred"] = (
        patient_df[prob_cols]
        .to_numpy()
        .argmax(axis=1)
        .astype(int)
    )

    return patient_df


# ============================================================
# 9. Ordinal losses
# ============================================================

class ExpectedValueOrdinalMAELoss(nn.Module):
    """
    Expected-value ordinal MAE:

        | E[C] - y |

    where:

        E[C] = sum_c c * p_c
    """

    def __init__(
        self,
        num_classes: int,
    ):
        super().__init__()

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        expected_class = (
            probabilities *
            self.class_values.unsqueeze(0)
        ).sum(dim=1)

        return torch.abs(
            expected_class -
            targets.float()
        ).mean()


class ExpectedOrdinalDistanceLoss(nn.Module):
    """
    Expected ordinal-distance loss:

        sum_c p_c * |y - c|^alpha
    """

    def __init__(
        self,
        num_classes: int,
        alpha: float = 1.0,
    ):
        super().__init__()

        self.alpha = float(alpha)

        self.register_buffer(
            "class_values",
            torch.arange(
                num_classes,
                dtype=torch.float32,
            ),
        )

    def forward(
        self,
        logits,
        targets,
    ):
        probabilities = torch.softmax(
            logits,
            dim=1,
        )

        targets = (
            targets
            .float()
            .unsqueeze(1)
        )

        distances = torch.abs(
            targets -
            self.class_values.unsqueeze(0)
        ) ** self.alpha

        return (
            probabilities *
            distances
        ).sum(dim=1).mean()


class CombinedLoss(nn.Module):
    def __init__(
        self,
        loss_type: str,
        ce_weight=None,
        label_smoothing: float = 0.0,
        num_classes: int = 4,
        ordinal_weight: float = 0.10,
        ordinal_alpha: float = 1.0,
    ):
        super().__init__()

        valid_loss_types = {
            "ce",
            "ce_expected_value_mae",
            "ce_expected_distance",
        }

        if loss_type not in valid_loss_types:
            raise ValueError(
                f"Unknown loss_type: {loss_type}"
            )

        self.loss_type = loss_type
        self.ordinal_weight = float(
            ordinal_weight
        )

        self.ce = nn.CrossEntropyLoss(
            weight=ce_weight,
            label_smoothing=label_smoothing,
        )

        if loss_type == "ce":
            self.ordinal = None

        elif loss_type == "ce_expected_value_mae":
            self.ordinal = (
                ExpectedValueOrdinalMAELoss(
                    num_classes=num_classes
                )
            )

        else:
            self.ordinal = (
                ExpectedOrdinalDistanceLoss(
                    num_classes=num_classes,
                    alpha=ordinal_alpha,
                )
            )

    def forward(
        self,
        logits,
        targets,
    ):
        ce_loss = self.ce(
            logits,
            targets,
        )

        if self.ordinal is None:
            ordinal_loss = torch.zeros(
                (),
                dtype=logits.dtype,
                device=logits.device,
            )

            total_loss = ce_loss

        else:
            ordinal_loss = self.ordinal(
                logits,
                targets,
            )

            total_loss = (
                ce_loss +
                self.ordinal_weight *
                ordinal_loss
            )

        return (
            total_loss,
            ce_loss.detach(),
            ordinal_loss.detach(),
        )


# ============================================================
# 10. Learning-rate scheduler
# ============================================================

def build_cosine_schedule_with_warmup(
    optimizer,
    warmup_epochs: int,
    total_epochs: int,
    min_lr_ratio: float = 0.05,
):
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (
                float(epoch + 1) /
                float(
                    max(
                        1,
                        warmup_epochs,
                    )
                )
            )

        progress = (
            float(
                epoch -
                warmup_epochs
            ) /
            float(
                max(
                    1,
                    total_epochs -
                    warmup_epochs,
                )
            )
        )

        cosine = 0.5 * (
            1.0 +
            math.cos(
                math.pi *
                progress
            )
        )

        return max(
            min_lr_ratio,
            cosine,
        )

    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda,
    )


# ============================================================
# 11. Mixed-precision compatibility helpers
# ============================================================

def make_grad_scaler(
    enabled: bool,
):
    try:
        return torch.amp.GradScaler(
            "cuda",
            enabled=enabled,
        )
    except Exception:
        return torch.cuda.amp.GradScaler(
            enabled=enabled
        )


def autocast_context(
    enabled: bool,
):
    try:
        return torch.amp.autocast(
            device_type="cuda",
            enabled=enabled,
        )
    except Exception:
        return torch.cuda.amp.autocast(
            enabled=enabled
        )


# ============================================================
# 12. Training and TTA inference
# ============================================================

def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device,
    scaler=None,
    grad_clip_norm=None,
):
    model.train()

    total_loss = 0.0

    y_true = []
    y_pred = []

    use_amp = (
        scaler is not None and
        scaler.is_enabled() and
        device.type == "cuda"
    )

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(images)

            loss, _, _ = criterion(
                logits,
                labels,
            )

        if use_amp:
            scaler.scale(
                loss
            ).backward()

            if grad_clip_norm is not None:
                scaler.unscale_(
                    optimizer
                )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip_norm,
                )

            scaler.step(
                optimizer
            )

            scaler.update()

        else:
            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip_norm,
                )

            optimizer.step()

        total_loss += (
            loss.item() *
            images.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss /
        len(loader.dataset)
    )

    return metrics


@torch.no_grad()
def probabilities_with_tta(
    model,
    images,
    original_logits,
    use_tta: bool = False,
    tta_horizontal_flip: bool = True,
):
    probability_list = [
        torch.softmax(
            original_logits,
            dim=1,
        )
    ]

    if (
        use_tta and
        tta_horizontal_flip
    ):
        flipped_images = torch.flip(
            images,
            dims=[3],
        )

        flipped_logits = model(
            flipped_images
        )

        probability_list.append(
            torch.softmax(
                flipped_logits,
                dim=1,
            )
        )

    return torch.stack(
        probability_list,
        dim=0,
    ).mean(dim=0)


@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion,
    device,
    num_classes: int = 4,
    use_tta: bool = False,
    tta_horizontal_flip: bool = True,
):
    model.eval()

    total_loss = 0.0

    y_true = []
    y_pred = []
    y_prob = []

    patient_ids = []
    patient_y_true = []
    paths = []

    use_amp = (
        device.type == "cuda"
    )

    for batch in loader:
        images = batch["image"].to(
            device,
            non_blocking=True,
        )

        labels = batch["label"].to(
            device,
            non_blocking=True,
        )

        with autocast_context(
            enabled=use_amp
        ):
            logits = model(images)

            loss, _, _ = criterion(
                logits,
                labels,
            )

            probabilities = probabilities_with_tta(
                model=model,
                images=images,
                original_logits=logits,
                use_tta=use_tta,
                tta_horizontal_flip=tta_horizontal_flip,
            )

        predictions = probabilities.argmax(
            dim=1
        )

        total_loss += (
            loss.item() *
            images.size(0)
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_pred.extend(
            predictions.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_prob.extend(
            probabilities.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        patient_ids.extend(
            batch["patient_id"]
            .cpu()
            .numpy()
            .tolist()
        )

        patient_y_true.extend(
            batch["patient_y_true"]
            .cpu()
            .numpy()
            .tolist()
        )

        paths.extend(
            batch["path"]
        )

    image_metrics = compute_metrics(
        y_true,
        y_pred,
        num_classes=num_classes,
    )

    image_metrics["loss"] = (
        total_loss /
        len(loader.dataset)
    )

    pred_df = pd.DataFrame({
        "path": paths,
        "patient_id": patient_ids,
        "y_true": y_true,
        "patient_y_true": patient_y_true,
        "y_pred": y_pred,
    })

    prob_cols = [
        f"prob_{class_index}"
        for class_index in range(
            num_classes
        )
    ]

    pred_df = pd.concat(
        [
            pred_df,
            pd.DataFrame(
                y_prob,
                columns=prob_cols,
            ),
        ],
        axis=1,
    )

    patient_df = (
        aggregate_mean_probabilities(
            pred_df,
            num_classes=num_classes,
        )
    )

    patient_metrics = compute_metrics(
        patient_df["y_true"],
        patient_df["y_pred"],
        num_classes=num_classes,
    )

    return (
        image_metrics,
        patient_metrics,
        pred_df,
        patient_df,
    )


# ============================================================
# 13. Patient-primary checkpoint selection
# ============================================================

def safe_metric_for_max(
    value,
):
    if value is None:
        return -np.inf

    if not np.isfinite(value):
        return -np.inf

    return float(value)


def safe_metric_for_min(
    value,
):
    if value is None:
        return np.inf

    if not np.isfinite(value):
        return np.inf

    return float(value)


def is_better_checkpoint(
    candidate_patient_metrics,
    best_patient_metrics,
    candidate_image_metrics,
    best_image_metrics,
    tolerance: float = 1e-12,
):
    """
    Lexicographic checkpoint selection:

      1. validation patient QWK: higher
      2. validation patient MAE: lower
      3. validation patient macro F1: higher
      4. validation image QWK: higher
    """

    if best_patient_metrics is None:
        return True

    candidate_values = [
        safe_metric_for_max(
            candidate_patient_metrics["qwk"]
        ),
        -safe_metric_for_min(
            candidate_patient_metrics["mae"]
        ),
        safe_metric_for_max(
            candidate_patient_metrics["macro_f1"]
        ),
        safe_metric_for_max(
            candidate_image_metrics["qwk"]
        ),
    ]

    best_values = [
        safe_metric_for_max(
            best_patient_metrics["qwk"]
        ),
        -safe_metric_for_min(
            best_patient_metrics["mae"]
        ),
        safe_metric_for_max(
            best_patient_metrics["macro_f1"]
        ),
        safe_metric_for_max(
            best_image_metrics["qwk"]
        ),
    ]

    for candidate_value, best_value in zip(
        candidate_values,
        best_values,
    ):
        if (
            candidate_value >
            best_value +
            tolerance
        ):
            return True

        if (
            candidate_value <
            best_value -
            tolerance
        ):
            return False

    return False


# ============================================================
# 14. Stratified patient bootstrap
# ============================================================

def stratified_patient_bootstrap(
    patient_df: pd.DataFrame,
    repetitions: int = 5000,
    seed: int = 2026,
    num_classes: int = 4,
):
    """
    Stratified patient bootstrap.

    Patients are resampled with replacement within each
    derived patient-level MES class.

    This estimates sampling uncertainty for OOF patient-level
    metrics. It does not capture model-training stochasticity.
    """

    required_columns = {
        "patient_id",
        "y_true",
        "y_pred",
    }

    missing_columns = (
        required_columns -
        set(patient_df.columns)
    )

    if missing_columns:
        raise ValueError(
            "patient_df is missing columns: "
            f"{sorted(missing_columns)}"
        )

    rng = np.random.default_rng(
        seed
    )

    class_frames = {
        class_index: (
            patient_df[
                patient_df["y_true"]
                ==
                class_index
            ]
            .reset_index(drop=True)
        )
        for class_index in range(
            num_classes
        )
    }

    for class_index, class_df in (
        class_frames.items()
    ):
        if len(class_df) == 0:
            raise ValueError(
                "Cannot perform stratified bootstrap: "
                f"no patients in class {class_index}"
            )

    bootstrap_rows = []

    for bootstrap_index in range(
        repetitions
    ):
        sampled_parts = []

        for class_index in range(
            num_classes
        ):
            class_df = class_frames[
                class_index
            ]

            sampled_indices = rng.integers(
                low=0,
                high=len(class_df),
                size=len(class_df),
            )

            sampled_class_df = (
                class_df
                .iloc[sampled_indices]
                .copy()
            )

            sampled_parts.append(
                sampled_class_df
            )

        sampled_df = pd.concat(
            sampled_parts,
            ignore_index=True,
        )

        metrics = compute_metrics(
            sampled_df["y_true"],
            sampled_df["y_pred"],
            num_classes=num_classes,
        )

        bootstrap_rows.append({
            "bootstrap_index": bootstrap_index,
            **metrics,
        })

    bootstrap_df = pd.DataFrame(
        bootstrap_rows
    )

    ci_rows = []

    for metric_name in [
        "qwk",
        "mae",
        "macro_f1",
        "accuracy",
    ]:
        valid_values = (
            bootstrap_df[metric_name]
            .dropna()
            .to_numpy()
        )

        if len(valid_values) == 0:
            lower_95 = np.nan
            upper_95 = np.nan

        else:
            lower_95 = float(
                np.percentile(
                    valid_values,
                    2.5,
                )
            )

            upper_95 = float(
                np.percentile(
                    valid_values,
                    97.5,
                )
            )

        ci_rows.append({
            "metric": metric_name,
            "lower_95": lower_95,
            "upper_95": upper_95,
            "n_valid_bootstrap": int(
                len(valid_values)
            ),
        })

    ci_df = pd.DataFrame(
        ci_rows
    )

    return (
        ci_df,
        bootstrap_df,
    )


print(
    "\nCell 2 completed successfully."
)

print(
    "Primary patient reference:",
    PRIMARY_REFERENCE_RULE,
)

print(
    "Official test data loaded:",
    False,
)

In [ ]:
# ============================================================
# Cell 3 preflight check
# ============================================================

assert PRIMARY_REFERENCE_RULE == "modal_low"

assert FOLD_ASSIGNMENT_PATH.exists(), (
    f"Missing fold assignment: {FOLD_ASSIGNMENT_PATH}"
)

assert REFERENCE_AUDIT_PATH.exists(), (
    f"Missing patient-reference audit: {REFERENCE_AUDIT_PATH}"
)

assert trainval_df["patient_id"].nunique() == 478
assert len(trainval_df) == 9590
assert trainval_df["cv_fold"].nunique() == 5
assert trainval_df["cv_fold"].isna().sum() == 0
assert trainval_df["patient_y_true"].isna().sum() == 0

forbidden_test_objects = [
    "TEST_PATH",
    "test_df",
    "test_loader",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
]

present_test_objects = [
    name
    for name in forbidden_test_objects
    if name in globals()
]

assert not present_test_objects, (
    "Official-test objects remain in memory: "
    f"{present_test_objects}"
)

print("Cell 3 preflight check passed.")
print("Official test data remain unavailable to this notebook.")

In [ ]:
# ============================================================
# CELL 3
# LIMUC ViT Loss Screening — Development-Only Training
# ------------------------------------------------------------
# Requirements:
#   - Complete revised Cell 1 has been run.
#   - Complete revised Cell 2 has been run.
#   - Cell 3 preflight check has passed.
#
# Main principles:
#   1. Official LIMUC test data are never loaded or evaluated.
#   2. The patient reference is fixed as modal_low.
#   3. The same frozen patient folds are used for every candidate.
#   4. Fold checkpoints are selected by:
#        a) validation patient QWK, higher
#        b) validation patient MAE, lower
#        c) validation patient macro F1, higher
#        d) validation image QWK, higher
#   5. Candidate losses are ranked by:
#        a) OOF patient QWK, higher
#        b) OOF patient MAE, lower
#        c) OOF patient macro F1, higher
#        d) OOF image QWK, higher
#   6. The top two are shortlisted, not declared final.
# ============================================================


# ============================================================
# 0. Imports
# ============================================================

import gc
import os
import json
import copy
import hashlib
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler

import timm

from IPython.display import display


# ============================================================
# 1. Execution controls
# ============================================================

# Skip fully completed experiment-seed combinations.
SKIP_COMPLETED_EXPERIMENTS = True

# Resume fully completed folds within a partially completed experiment.
RESUME_COMPLETED_FOLDS = True

# These should have been defined in Cell 1.
# Initial screening recommendation:
# TRAIN_SEEDS = [42]
#
# EXPERIMENTS_TO_RUN = [
#     "vit_L1_ce_only",
#     "vit_L2_expected_value_ordinal_mae_w010",
#     "vit_L3_expected_ordinal_distance_a1_w010",
#     "vit_L4_expected_ordinal_distance_a2_w010",
#     "vit_L5_expected_ordinal_distance_a1_w020",
# ]

print("Experiments to run:")
for experiment_name in EXPERIMENTS_TO_RUN:
    print(" -", experiment_name)

print("Training seeds:", TRAIN_SEEDS)


# ============================================================
# 2. Complete preflight validation
# ============================================================

required_objects = [
    "BASE_SAVE_ROOT",
    "BASE_CONFIG",
    "EXPERIMENT_PRESETS",
    "EXPERIMENTS_TO_RUN",
    "TRAIN_SEEDS",
    "trainval_df",
    "PRIMARY_REFERENCE_RULE",
    "FOLD_ASSIGNMENT_PATH",
    "REFERENCE_AUDIT_PATH",
    "reference_audit_check",
    "seed_everything",
    "seed_worker",
    "LIMUCDataset",
    "build_train_transform",
    "build_eval_transform",
    "CombinedLoss",
    "build_cosine_schedule_with_warmup",
    "make_grad_scaler",
    "train_one_epoch",
    "evaluate",
    "is_better_checkpoint",
    "aggregate_mean_probabilities",
    "compute_metrics",
    "make_confusion_df",
    "stratified_patient_bootstrap",
]

missing_objects = [
    object_name
    for object_name in required_objects
    if object_name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Cell 1 and Cell 2 must be run before Cell 3. "
        f"Missing objects: {missing_objects}"
    )

forbidden_test_objects = [
    "TEST_PATH",
    "test_df",
    "test_loader",
    "ensemble_test_df",
    "ensemble_test_patient_df",
    "test_fold_prob_dfs",
    "all_test_fold_prob_df",
    "ensemble_test_metrics",
    "ensemble_test_patient_metrics",
]

present_test_objects = [
    object_name
    for object_name in forbidden_test_objects
    if object_name in globals()
]

if present_test_objects:
    raise RuntimeError(
        "Official-test objects remain in the runtime: "
        f"{present_test_objects}\n"
        "Restart the runtime and run only the revised Cells 1–3."
    )

if PRIMARY_REFERENCE_RULE != "modal_low":
    raise RuntimeError(
        "The primary patient reference must remain modal_low "
        "for this frozen screening analysis."
    )

BASE_SAVE_ROOT = Path(BASE_SAVE_ROOT)

if not FOLD_ASSIGNMENT_PATH.exists():
    raise FileNotFoundError(
        f"Frozen fold assignment is missing:\n{FOLD_ASSIGNMENT_PATH}"
    )

if not REFERENCE_AUDIT_PATH.exists():
    raise FileNotFoundError(
        f"Patient-reference audit is missing:\n{REFERENCE_AUDIT_PATH}"
    )

if len(trainval_df) != 9590:
    raise RuntimeError(
        f"Expected 9,590 development images, found {len(trainval_df)}."
    )

if trainval_df["patient_id"].nunique() != 478:
    raise RuntimeError(
        "Expected 478 development patients, found "
        f"{trainval_df['patient_id'].nunique()}."
    )

if trainval_df["cv_fold"].nunique() != BASE_CONFIG["num_folds"]:
    raise RuntimeError(
        "Unexpected number of development folds."
    )

if trainval_df["cv_fold"].isna().any():
    raise RuntimeError(
        "At least one development image has no fold assignment."
    )

if trainval_df["patient_y_true"].isna().any():
    raise RuntimeError(
        "At least one development image has no patient reference."
    )

unknown_experiments = (
    set(EXPERIMENTS_TO_RUN)
    -
    set(EXPERIMENT_PRESETS)
)

if unknown_experiments:
    raise KeyError(
        "Unknown experiments in EXPERIMENTS_TO_RUN: "
        f"{sorted(unknown_experiments)}"
    )

if len(TRAIN_SEEDS) == 0:
    raise ValueError("TRAIN_SEEDS cannot be empty.")

print("\nCell 3 preflight check passed.")
print("Primary patient reference:", PRIMARY_REFERENCE_RULE)
print("Official test data loaded:", False)
print("Official test data evaluated:", False)


# ============================================================
# 3. Device
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("\nDevice:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# 4. Hash and serialization helpers
# ============================================================

def file_sha256(path: Path) -> str:
    hasher = hashlib.sha256()

    with open(path, "rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            hasher.update(block)

    return hasher.hexdigest()


def dictionary_sha256(dictionary: dict) -> str:
    serialized = json.dumps(
        dictionary,
        sort_keys=True,
        separators=(",", ":"),
        default=str,
    ).encode("utf-8")

    return hashlib.sha256(serialized).hexdigest()


def state_dict_to_cpu(model) -> dict:
    return {
        key: value.detach().cpu()
        for key, value in model.state_dict().items()
    }


def load_torch_checkpoint(
    checkpoint_path: Path,
    map_location,
):
    try:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            checkpoint_path,
            map_location=map_location,
        )


# ============================================================
# 5. Freeze development-only analysis plan
# ============================================================

immutable_analysis_plan = {
    "study_stage": "Development-only ViT loss screening",

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "development_images": int(len(trainval_df)),
    "development_patients": int(
        trainval_df["patient_id"].nunique()
    ),

    "primary_patient_reference": PRIMARY_REFERENCE_RULE,

    "primary_patient_reference_definition": (
        "Modal image-level MES; when multiple modal classes "
        "are tied, the lower class is selected."
    ),

    "patient_prediction_aggregation": "mean_probability",

    "fold_assignment_file": str(FOLD_ASSIGNMENT_PATH),
    "fold_assignment_sha256": file_sha256(
        FOLD_ASSIGNMENT_PATH
    ),

    "patient_reference_audit_file": str(
        REFERENCE_AUDIT_PATH
    ),
    "patient_reference_audit_sha256": file_sha256(
        REFERENCE_AUDIT_PATH
    ),

    "candidate_experiments": list(
        EXPERIMENTS_TO_RUN
    ),

    "training_seeds": [
        int(seed)
        for seed in TRAIN_SEEDS
    ],

    "checkpoint_selection_hierarchy": [
        "Validation patient QWK: higher",
        "Validation patient MAE: lower",
        "Validation patient macro F1: higher",
        "Validation image QWK: higher",
    ],

    "experiment_ranking_hierarchy": [
        "OOF patient QWK: higher",
        "OOF patient MAE: lower",
        "OOF patient macro F1: higher",
        "OOF image QWK: higher",
    ],

    "screening_interpretation": (
        "The top-ranked single-seed candidate is a screening "
        "result. The top two candidates are shortlisted for "
        "multi-seed confirmation and are not treated as final."
    ),
}

analysis_plan_fingerprint = dictionary_sha256(
    immutable_analysis_plan
)

analysis_plan = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    **immutable_analysis_plan,
}

analysis_plan_path = (
    BASE_SAVE_ROOT
    /
    "analysis_plan_frozen_before_training.json"
)

if analysis_plan_path.exists():
    with open(
        analysis_plan_path,
        "r",
        encoding="utf-8",
    ) as file:
        existing_plan = json.load(file)

    existing_fingerprint = existing_plan.get(
        "analysis_plan_fingerprint"
    )

    if existing_fingerprint is None:
        existing_immutable_plan = {
            key: existing_plan.get(key)
            for key in immutable_analysis_plan
        }

        existing_fingerprint = dictionary_sha256(
            existing_immutable_plan
        )

    if existing_fingerprint != analysis_plan_fingerprint:
        raise RuntimeError(
            "The current analysis configuration differs from "
            "the previously frozen analysis plan.\n"
            "Do not overwrite the existing revision directory. "
            "Archive it and use a new output directory."
        )

    print(
        "\nExisting frozen analysis plan validated:",
        analysis_plan_path,
    )

else:
    with open(
        analysis_plan_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            analysis_plan,
            file,
            indent=2,
        )

    print(
        "\nSaved frozen analysis plan to:",
        analysis_plan_path,
    )

print(
    "Analysis-plan fingerprint:",
    analysis_plan_fingerprint,
)


# ============================================================
# 6. Class-weight helper
# ============================================================

def calculate_class_weights(
    train_df: pd.DataFrame,
    num_classes: int,
    device,
):
    class_counts = (
        train_df["label"]
        .value_counts()
        .reindex(
            range(num_classes),
            fill_value=0,
        )
        .sort_index()
    )

    absent_classes = (
        class_counts[
            class_counts == 0
        ]
        .index
        .tolist()
    )

    if absent_classes:
        raise ValueError(
            "At least one image-level class is absent "
            "from the training fold: "
            f"{absent_classes}"
        )

    total_images = int(
        class_counts.sum()
    )

    weights = (
        total_images
        /
        (
            num_classes
            *
            class_counts
        )
    )

    weight_tensor = torch.tensor(
        weights.to_numpy(),
        dtype=torch.float32,
        device=device,
    )

    return (
        weight_tensor,
        class_counts.to_dict(),
        weights.to_dict(),
    )


# ============================================================
# 7. Load a completed fold
# ============================================================

def load_completed_fold(
    fold_dir: Path,
    expected_config_fingerprint: str,
    expected_analysis_plan_fingerprint: str,
):
    fold_result_path = (
        fold_dir
        /
        "fold_result.json"
    )

    image_prediction_path = (
        fold_dir
        /
        "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_dir
        /
        "validation_patient_predictions.csv"
    )

    checkpoint_path = (
        fold_dir
        /
        "best_checkpoint.pt"
    )

    required_paths = [
        fold_result_path,
        image_prediction_path,
        patient_prediction_path,
        checkpoint_path,
    ]

    if not all(path.exists() for path in required_paths):
        return None

    with open(
        fold_result_path,
        "r",
        encoding="utf-8",
    ) as file:
        fold_metadata = json.load(file)

    if (
        fold_metadata.get("config_fingerprint")
        !=
        expected_config_fingerprint
    ):
        raise RuntimeError(
            "A completed fold was found, but its configuration "
            "does not match the current experiment:\n"
            f"{fold_dir}"
        )

    if (
        fold_metadata.get("analysis_plan_fingerprint")
        !=
        expected_analysis_plan_fingerprint
    ):
        raise RuntimeError(
            "A completed fold was found, but its analysis plan "
            "does not match the current frozen plan:\n"
            f"{fold_dir}"
        )

    image_prediction_df = pd.read_csv(
        image_prediction_path
    )

    patient_prediction_df = pd.read_csv(
        patient_prediction_path
    )

    return {
        "fold": int(fold_metadata["fold"]),
        "training_seed": int(
            fold_metadata["training_seed"]
        ),
        "best_epoch": int(
            fold_metadata["best_epoch"]
        ),

        "image_metrics": fold_metadata[
            "image_metrics"
        ],

        "patient_metrics": fold_metadata[
            "patient_metrics"
        ],

        "val_patient_class_counts": (
            fold_metadata[
                "val_patient_class_counts"
            ]
        ),

        "val_pred_df": image_prediction_df,
        "val_patient_df": patient_prediction_df,

        "checkpoint_path": str(
            checkpoint_path
        ),

        "history_path": str(
            fold_dir
            /
            "history.csv"
        ),

        "pretrained_cfg": fold_metadata.get(
            "pretrained_cfg",
            {},
        ),
    }


# ============================================================
# 8. Run one fold
# ============================================================

def run_one_fold(
    fold: int,
    trainval_df: pd.DataFrame,
    config: dict,
    save_dir: Path,
    device,
    training_seed: int,
    analysis_plan_fingerprint: str,
):
    fold_seed = (
        int(training_seed)
        +
        int(fold)
    )

    config_fingerprint = dictionary_sha256(
        config
    )

    fold_dir = (
        save_dir
        /
        f"fold_{fold}"
    )

    fold_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    if RESUME_COMPLETED_FOLDS:
        completed_result = load_completed_fold(
            fold_dir=fold_dir,
            expected_config_fingerprint=(
                config_fingerprint
            ),
            expected_analysis_plan_fingerprint=(
                analysis_plan_fingerprint
            ),
        )

        if completed_result is not None:
            print(
                "\nLoaded completed fold:",
                config["experiment_name"],
                "| seed",
                training_seed,
                "| fold",
                fold,
            )

            return completed_result

    seed_everything(
        fold_seed
    )

    train_df = (
        trainval_df[
            trainval_df["cv_fold"]
            !=
            fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    val_df = (
        trainval_df[
            trainval_df["cv_fold"]
            ==
            fold
        ]
        .copy()
        .reset_index(drop=True)
    )

    train_patient_ids = set(
        train_df["patient_id"].unique()
    )

    val_patient_ids = set(
        val_df["patient_id"].unique()
    )

    patient_overlap = (
        train_patient_ids
        &
        val_patient_ids
    )

    if patient_overlap:
        raise RuntimeError(
            f"Patient leakage detected in fold {fold}: "
            f"{sorted(patient_overlap)[:20]}"
        )

    val_patient_distribution = (
        val_df[
            [
                "patient_id",
                "patient_y_true",
            ]
        ]
        .drop_duplicates()
        ["patient_y_true"]
        .value_counts()
        .reindex(
            range(config["num_classes"]),
            fill_value=0,
        )
        .sort_index()
    )

    print("\n" + "=" * 80)

    print(
        f"{config['experiment_name']} | "
        f"seed={training_seed} | "
        f"fold={fold + 1}/{config['num_folds']}"
    )

    print("=" * 80)

    print("Training images:", len(train_df))
    print("Validation images:", len(val_df))
    print("Training patients:", len(train_patient_ids))
    print("Validation patients:", len(val_patient_ids))

    for mayo_class in range(
        config["num_classes"]
    ):
        print(
            f"Validation patients, Mayo {mayo_class}:",
            int(
                val_patient_distribution.loc[
                    mayo_class
                ]
            ),
        )

    train_transform = build_train_transform(
        config["image_size"]
    )

    eval_transform = build_eval_transform(
        config["image_size"]
    )

    train_dataset = LIMUCDataset(
        train_df,
        transform=train_transform,
    )

    val_dataset = LIMUCDataset(
        val_df,
        transform=eval_transform,
    )

    generator = torch.Generator()
    generator.manual_seed(fold_seed)

    pin_memory = (
        device.type == "cuda"
    )

    if config["use_weighted_sampler"]:
        class_counts_for_sampler = (
            train_df["label"]
            .value_counts()
        )

        sample_weights = (
            train_df["label"]
            .map(
                lambda label: (
                    1.0
                    /
                    class_counts_for_sampler[
                        label
                    ]
                )
            )
            .to_numpy()
        )

        try:
            sampler = WeightedRandomSampler(
                weights=torch.DoubleTensor(
                    sample_weights
                ),
                num_samples=len(
                    sample_weights
                ),
                replacement=True,
                generator=generator,
            )

        except TypeError:
            sampler = WeightedRandomSampler(
                weights=torch.DoubleTensor(
                    sample_weights
                ),
                num_samples=len(
                    sample_weights
                ),
                replacement=True,
            )

        train_loader = DataLoader(
            train_dataset,
            batch_size=config["batch_size"],
            sampler=sampler,
            shuffle=False,
            num_workers=config["num_workers"],
            pin_memory=pin_memory,
            drop_last=False,
            worker_init_fn=seed_worker,
            generator=generator,
        )

    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=config["batch_size"],
            shuffle=True,
            num_workers=config["num_workers"],
            pin_memory=pin_memory,
            drop_last=False,
            worker_init_fn=seed_worker,
            generator=generator,
        )

    val_loader = DataLoader(
        val_dataset,
        batch_size=config["batch_size"],
        shuffle=False,
        num_workers=config["num_workers"],
        pin_memory=pin_memory,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=generator,
    )

    model = timm.create_model(
        config["model_name"],
        pretrained=config["pretrained"],
        num_classes=config["num_classes"],
        drop_rate=config["drop_rate"],
        drop_path_rate=config["drop_path_rate"],
    ).to(device)

    pretrained_cfg = copy.deepcopy(
        getattr(
            model,
            "pretrained_cfg",
            {},
        )
    )

    ce_weights = None
    image_class_counts = {}
    image_class_weights = {}

    if config["use_class_weights_in_ce"]:
        (
            ce_weights,
            image_class_counts,
            image_class_weights,
        ) = calculate_class_weights(
            train_df=train_df,
            num_classes=config["num_classes"],
            device=device,
        )

    criterion = CombinedLoss(
        loss_type=config["loss_type"],
        ce_weight=ce_weights,
        label_smoothing=config[
            "label_smoothing"
        ],
        num_classes=config["num_classes"],
        ordinal_weight=config[
            "ordinal_loss_weight"
        ],
        ordinal_alpha=config[
            "ordinal_alpha"
        ],
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"],
    )

    scheduler = (
        build_cosine_schedule_with_warmup(
            optimizer=optimizer,
            warmup_epochs=config[
                "warmup_epochs"
            ],
            total_epochs=config["epochs"],
            min_lr_ratio=config[
                "min_lr_ratio"
            ],
        )
    )

    scaler = make_grad_scaler(
        enabled=(
            config["use_mixed_precision"]
            and
            device.type == "cuda"
        )
    )

    checkpoint_path = (
        fold_dir
        /
        "best_checkpoint.pt"
    )

    history_path = (
        fold_dir
        /
        "history.csv"
    )

    image_prediction_path = (
        fold_dir
        /
        "validation_image_predictions.csv"
    )

    patient_prediction_path = (
        fold_dir
        /
        "validation_patient_predictions.csv"
    )

    fold_result_path = (
        fold_dir
        /
        "fold_result.json"
    )

    history = []

    best_epoch = -1
    best_patient_metrics = None
    best_image_metrics = None
    patience_counter = 0

    for epoch in range(
        1,
        config["epochs"] + 1,
    ):
        train_metrics = train_one_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            scaler=scaler,
            grad_clip_norm=config[
                "grad_clip_norm"
            ],
        )

        (
            val_image_metrics,
            val_patient_metrics,
            val_prediction_df,
            val_patient_df,
        ) = evaluate(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
            num_classes=config["num_classes"],
            use_tta=config["use_tta"],
            tta_horizontal_flip=config[
                "tta_horizontal_flip"
            ],
        )

        current_lr = (
            optimizer.param_groups[0]["lr"]
        )

        history.append({
            "epoch": int(epoch),
            "lr": float(current_lr),

            "train_loss": float(
                train_metrics["loss"]
            ),
            "train_image_qwk": float(
                train_metrics["qwk"]
            ),
            "train_image_mae": float(
                train_metrics["mae"]
            ),
            "train_image_macro_f1": float(
                train_metrics["macro_f1"]
            ),
            "train_image_accuracy": float(
                train_metrics["accuracy"]
            ),

            "val_loss": float(
                val_image_metrics["loss"]
            ),
            "val_image_qwk": float(
                val_image_metrics["qwk"]
            ),
            "val_image_mae": float(
                val_image_metrics["mae"]
            ),
            "val_image_macro_f1": float(
                val_image_metrics[
                    "macro_f1"
                ]
            ),
            "val_image_accuracy": float(
                val_image_metrics[
                    "accuracy"
                ]
            ),

            "val_patient_qwk": float(
                val_patient_metrics["qwk"]
            ),
            "val_patient_mae": float(
                val_patient_metrics["mae"]
            ),
            "val_patient_macro_f1": float(
                val_patient_metrics[
                    "macro_f1"
                ]
            ),
            "val_patient_accuracy": float(
                val_patient_metrics[
                    "accuracy"
                ]
            ),
        })

        print(
            f"Epoch {epoch:02d}/{config['epochs']} | "
            f"lr={current_lr:.6f} | "
            f"val patient QWK="
            f"{val_patient_metrics['qwk']:.4f} | "
            f"val patient MAE="
            f"{val_patient_metrics['mae']:.4f} | "
            f"val patient F1="
            f"{val_patient_metrics['macro_f1']:.4f} | "
            f"val image QWK="
            f"{val_image_metrics['qwk']:.4f}"
        )

        improved = is_better_checkpoint(
            candidate_patient_metrics=(
                val_patient_metrics
            ),
            best_patient_metrics=(
                best_patient_metrics
            ),
            candidate_image_metrics=(
                val_image_metrics
            ),
            best_image_metrics=(
                best_image_metrics
            ),
        )

        if improved:
            best_epoch = int(epoch)

            best_patient_metrics = (
                copy.deepcopy(
                    val_patient_metrics
                )
            )

            best_image_metrics = (
                copy.deepcopy(
                    val_image_metrics
                )
            )

            patience_counter = 0

            checkpoint_payload = {
                "epoch": int(epoch),
                "fold": int(fold),
                "training_seed": int(
                    training_seed
                ),

                "analysis_plan_fingerprint": (
                    analysis_plan_fingerprint
                ),

                "config_fingerprint": (
                    config_fingerprint
                ),

                "config": copy.deepcopy(
                    config
                ),

                "model_state_dict": (
                    state_dict_to_cpu(model)
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "best_patient_metrics": (
                    best_patient_metrics
                ),

                "best_image_metrics": (
                    best_image_metrics
                ),

                "pretrained_cfg": (
                    pretrained_cfg
                ),
            }

            torch.save(
                checkpoint_payload,
                checkpoint_path,
            )

            val_prediction_df.to_csv(
                image_prediction_path,
                index=False,
            )

            val_patient_df.to_csv(
                patient_prediction_path,
                index=False,
            )

            print(
                "  Saved checkpoint: "
                f"patient QWK="
                f"{val_patient_metrics['qwk']:.4f}, "
                f"MAE="
                f"{val_patient_metrics['mae']:.4f}"
            )

        else:
            patience_counter += 1

            print(
                "  No improvement. "
                f"Patience "
                f"{patience_counter}/"
                f"{config['early_stop_patience']}"
            )

        pd.DataFrame(
            history
        ).to_csv(
            history_path,
            index=False,
        )

        scheduler.step()

        if (
            patience_counter
            >=
            config["early_stop_patience"]
        ):
            print("Early stopping triggered.")
            break

    if not checkpoint_path.exists():
        raise RuntimeError(
            "No checkpoint was saved for "
            f"{config['experiment_name']}, "
            f"seed {training_seed}, fold {fold}."
        )

    checkpoint = load_torch_checkpoint(
        checkpoint_path,
        map_location=device,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    (
        final_image_metrics,
        final_patient_metrics,
        final_prediction_df,
        final_patient_df,
    ) = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
        num_classes=config["num_classes"],
        use_tta=config["use_tta"],
        tta_horizontal_flip=config[
            "tta_horizontal_flip"
        ],
    )

    final_prediction_df["fold"] = int(fold)
    final_prediction_df["training_seed"] = int(
        training_seed
    )

    final_patient_df["fold"] = int(fold)
    final_patient_df["training_seed"] = int(
        training_seed
    )

    final_prediction_df.to_csv(
        image_prediction_path,
        index=False,
    )

    final_patient_df.to_csv(
        patient_prediction_path,
        index=False,
    )

    val_patient_class_counts = {
        str(mayo_class): int(
            val_patient_distribution.loc[
                mayo_class
            ]
        )
        for mayo_class in range(
            config["num_classes"]
        )
    }

    fold_metadata = {
        "fold": int(fold),
        "training_seed": int(
            training_seed
        ),
        "best_epoch": int(best_epoch),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),

        "config_fingerprint": (
            config_fingerprint
        ),

        "image_metrics": {
            key: float(value)
            for key, value in (
                final_image_metrics.items()
            )
        },

        "patient_metrics": {
            key: float(value)
            for key, value in (
                final_patient_metrics.items()
            )
        },

        "val_patient_class_counts": (
            val_patient_class_counts
        ),

        "training_image_class_counts": {
            str(key): int(value)
            for key, value in (
                image_class_counts.items()
            )
        },

        "training_image_class_weights": {
            str(key): float(value)
            for key, value in (
                image_class_weights.items()
            )
        },

        "pretrained_cfg": pretrained_cfg,

        "checkpoint_path": str(
            checkpoint_path
        ),
        "history_path": str(
            history_path
        ),
        "image_prediction_path": str(
            image_prediction_path
        ),
        "patient_prediction_path": str(
            patient_prediction_path
        ),
    }

    with open(
        fold_result_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            fold_metadata,
            file,
            indent=2,
            default=str,
        )

    result = {
        "fold": int(fold),
        "training_seed": int(
            training_seed
        ),
        "best_epoch": int(best_epoch),

        "image_metrics": {
            key: float(value)
            for key, value in (
                final_image_metrics.items()
            )
        },

        "patient_metrics": {
            key: float(value)
            for key, value in (
                final_patient_metrics.items()
            )
        },

        "val_patient_class_counts": (
            val_patient_class_counts
        ),

        "val_pred_df": (
            final_prediction_df
        ),

        "val_patient_df": (
            final_patient_df
        ),

        "checkpoint_path": str(
            checkpoint_path
        ),

        "history_path": str(
            history_path
        ),

        "pretrained_cfg": (
            pretrained_cfg
        ),
    }

    del model
    del criterion
    del optimizer
    del scheduler
    del scaler
    del checkpoint

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


# ============================================================
# 9. CI extraction helper
# ============================================================

def extract_ci_bounds(
    bootstrap_ci_df: pd.DataFrame,
    metric_name: str,
):
    metric_rows = (
        bootstrap_ci_df[
            bootstrap_ci_df["metric"]
            ==
            metric_name
        ]
    )

    if len(metric_rows) != 1:
        raise RuntimeError(
            "Expected one bootstrap CI row for "
            f"{metric_name}, found {len(metric_rows)}."
        )

    metric_row = metric_rows.iloc[0]

    return (
        float(metric_row["lower_95"]),
        float(metric_row["upper_95"]),
    )


# ============================================================
# 10. Run one complete candidate experiment
# ============================================================

def run_experiment(
    experiment_name: str,
    preset: dict,
    training_seed: int,
):
    config = copy.deepcopy(
        BASE_CONFIG
    )

    config.update(preset)

    config["experiment_name"] = (
        experiment_name
    )

    config["training_seed"] = int(
        training_seed
    )

    config["primary_patient_reference"] = (
        PRIMARY_REFERENCE_RULE
    )

    config["official_test_loaded"] = False
    config["official_test_evaluated"] = False

    config["analysis_plan_fingerprint"] = (
        analysis_plan_fingerprint
    )

    config_fingerprint = dictionary_sha256(
        config
    )

    save_dir = (
        BASE_SAVE_ROOT
        /
        experiment_name
        /
        f"seed_{training_seed}"
    )

    save_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    result_row_path = (
        save_dir
        /
        "result_row.json"
    )

    if (
        SKIP_COMPLETED_EXPERIMENTS
        and
        result_row_path.exists()
    ):
        with open(
            result_row_path,
            "r",
            encoding="utf-8",
        ) as file:
            completed_result = json.load(file)

        if (
            completed_result.get(
                "config_fingerprint"
            )
            !=
            config_fingerprint
        ):
            raise RuntimeError(
                "A completed experiment exists, but its "
                "configuration does not match the current one:\n"
                f"{save_dir}"
            )

        if (
            completed_result.get(
                "analysis_plan_fingerprint"
            )
            !=
            analysis_plan_fingerprint
        ):
            raise RuntimeError(
                "A completed experiment exists, but its "
                "analysis plan does not match the frozen plan:\n"
                f"{save_dir}"
            )

        print(
            "\nLoaded completed experiment:",
            experiment_name,
            "| seed",
            training_seed,
        )

        return completed_result

    config_path = (
        save_dir
        /
        "config_used.json"
    )

    with open(
        config_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            config,
            file,
            indent=2,
        )

    fold_results = []
    oof_image_parts = []

    for fold in range(
        config["num_folds"]
    ):
        fold_result = run_one_fold(
            fold=fold,
            trainval_df=trainval_df,
            config=config,
            save_dir=save_dir,
            device=device,
            training_seed=int(
                training_seed
            ),
            analysis_plan_fingerprint=(
                analysis_plan_fingerprint
            ),
        )

        fold_results.append(
            fold_result
        )

        oof_image_parts.append(
            fold_result["val_pred_df"]
        )

    oof_image_df = (
        pd.concat(
            oof_image_parts,
            ignore_index=True,
        )
        .sort_values("path")
        .reset_index(drop=True)
    )

    if len(oof_image_df) != len(
        trainval_df
    ):
        raise RuntimeError(
            "OOF row count does not match the "
            "development image count.\n"
            f"OOF rows: {len(oof_image_df)}\n"
            f"Development images: {len(trainval_df)}"
        )

    if (
        oof_image_df["path"].nunique()
        !=
        len(trainval_df)
    ):
        raise RuntimeError(
            "OOF image paths are duplicated or missing."
        )

    if (
        oof_image_df
        .groupby("patient_id")["fold"]
        .nunique()
        .max()
        !=
        1
    ):
        raise RuntimeError(
            "A patient appears in more than one "
            "OOF validation fold."
        )

    oof_patient_df = (
        aggregate_mean_probabilities(
            oof_image_df,
            num_classes=config[
                "num_classes"
            ],
        )
    )

    reference_metadata_columns = [
        "patient_id",
        "modal_low",
        "modal_high",
        "median_low",
        "median_high",
        "maximum_label",
        "n_images",
        "n_unique_labels",
        "label_range",
        "modal_tie",
        "majority_fraction",
        "label_entropy",
    ]

    available_reference_columns = [
        column
        for column in (
            reference_metadata_columns
        )
        if column in (
            reference_audit_check.columns
        )
    ]

    oof_patient_df = (
        oof_patient_df
        .merge(
            reference_audit_check[
                available_reference_columns
            ],
            on="patient_id",
            how="left",
            validate="one_to_one",
            suffixes=(
                "",
                "_audit",
            ),
        )
    )

    if (
        "n_images_audit"
        in
        oof_patient_df.columns
    ):
        image_count_matches = (
            oof_patient_df["n_images"]
            ==
            oof_patient_df[
                "n_images_audit"
            ]
        )

        if not image_count_matches.all():
            raise RuntimeError(
                "OOF patient image counts do not match "
                "the patient-reference audit."
            )

        oof_patient_df = (
            oof_patient_df.drop(
                columns=[
                    "n_images_audit"
                ]
            )
        )

    reference_matches = (
        oof_patient_df["y_true"].astype(int)
        ==
        oof_patient_df[
            PRIMARY_REFERENCE_RULE
        ].astype(int)
    )

    if not reference_matches.all():
        raise RuntimeError(
            "OOF patient y_true does not match "
            "the frozen modal-low reference."
        )

    oof_image_metrics = compute_metrics(
        oof_image_df["y_true"],
        oof_image_df["y_pred"],
        num_classes=config["num_classes"],
    )

    oof_patient_metrics = compute_metrics(
        oof_patient_df["y_true"],
        oof_patient_df["y_pred"],
        num_classes=config["num_classes"],
    )

    oof_image_path = (
        save_dir
        /
        "oof_image_predictions.csv"
    )

    oof_patient_path = (
        save_dir
        /
        "oof_patient_predictions.csv"
    )

    oof_image_df.to_csv(
        oof_image_path,
        index=False,
    )

    oof_patient_df.to_csv(
        oof_patient_path,
        index=False,
    )

    make_confusion_df(
        oof_image_df["y_true"],
        oof_image_df["y_pred"],
        num_classes=config["num_classes"],
    ).to_csv(
        save_dir
        /
        "oof_image_confusion_matrix.csv"
    )

    make_confusion_df(
        oof_patient_df["y_true"],
        oof_patient_df["y_pred"],
        num_classes=config["num_classes"],
    ).to_csv(
        save_dir
        /
        "oof_patient_confusion_matrix.csv"
    )

    (
        bootstrap_ci_df,
        bootstrap_samples_df,
    ) = stratified_patient_bootstrap(
        patient_df=oof_patient_df,
        repetitions=config[
            "bootstrap_repetitions"
        ],
        seed=(
            config["bootstrap_seed"]
            +
            int(training_seed)
        ),
        num_classes=config["num_classes"],
    )

    point_estimate_map = {
        "qwk": oof_patient_metrics["qwk"],
        "mae": oof_patient_metrics["mae"],
        "macro_f1": (
            oof_patient_metrics["macro_f1"]
        ),
        "accuracy": (
            oof_patient_metrics["accuracy"]
        ),
    }

    bootstrap_ci_df.insert(
        1,
        "point_estimate",
        bootstrap_ci_df["metric"].map(
            point_estimate_map
        ),
    )

    bootstrap_ci_path = (
        save_dir
        /
        "oof_patient_bootstrap_ci.csv"
    )

    bootstrap_samples_path = (
        save_dir
        /
        "oof_patient_bootstrap_samples.csv"
    )

    bootstrap_ci_df.to_csv(
        bootstrap_ci_path,
        index=False,
    )

    bootstrap_samples_df.to_csv(
        bootstrap_samples_path,
        index=False,
    )

    (
        qwk_ci_low,
        qwk_ci_high,
    ) = extract_ci_bounds(
        bootstrap_ci_df,
        "qwk",
    )

    (
        mae_ci_low,
        mae_ci_high,
    ) = extract_ci_bounds(
        bootstrap_ci_df,
        "mae",
    )

    (
        macro_f1_ci_low,
        macro_f1_ci_high,
    ) = extract_ci_bounds(
        bootstrap_ci_df,
        "macro_f1",
    )

    (
        accuracy_ci_low,
        accuracy_ci_high,
    ) = extract_ci_bounds(
        bootstrap_ci_df,
        "accuracy",
    )

    fold_summary_rows = []

    for fold_result in fold_results:
        row = {
            "fold": int(
                fold_result["fold"]
            ),

            "training_seed": int(
                fold_result[
                    "training_seed"
                ]
            ),

            "best_epoch": int(
                fold_result["best_epoch"]
            ),

            "val_image_qwk": float(
                fold_result[
                    "image_metrics"
                ]["qwk"]
            ),

            "val_image_mae": float(
                fold_result[
                    "image_metrics"
                ]["mae"]
            ),

            "val_image_macro_f1": float(
                fold_result[
                    "image_metrics"
                ]["macro_f1"]
            ),

            "val_image_accuracy": float(
                fold_result[
                    "image_metrics"
                ]["accuracy"]
            ),

            "val_patient_qwk": float(
                fold_result[
                    "patient_metrics"
                ]["qwk"]
            ),

            "val_patient_mae": float(
                fold_result[
                    "patient_metrics"
                ]["mae"]
            ),

            "val_patient_macro_f1": float(
                fold_result[
                    "patient_metrics"
                ]["macro_f1"]
            ),

            "val_patient_accuracy": float(
                fold_result[
                    "patient_metrics"
                ]["accuracy"]
            ),

            "checkpoint_path": (
                fold_result[
                    "checkpoint_path"
                ]
            ),

            "history_path": (
                fold_result["history_path"]
            ),
        }

        for mayo_class in range(
            config["num_classes"]
        ):
            row[
                f"val_patient_count_mayo_{mayo_class}"
            ] = int(
                fold_result[
                    "val_patient_class_counts"
                ][str(mayo_class)]
            )

        fold_summary_rows.append(row)

    fold_summary_df = pd.DataFrame(
        fold_summary_rows
    )

    fold_summary_path = (
        save_dir
        /
        "fold_summary.csv"
    )

    fold_summary_df.to_csv(
        fold_summary_path,
        index=False,
    )

    pretrained_cfg = (
        fold_results[0]["pretrained_cfg"]
    )

    pretrained_cfg_path = (
        save_dir
        /
        "pretrained_cfg.json"
    )

    with open(
        pretrained_cfg_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            pretrained_cfg,
            file,
            indent=2,
            default=str,
        )

    summary = {
        "experiment_name": experiment_name,
        "display_name": preset["display_name"],
        "training_seed": int(training_seed),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),
        "config_fingerprint": (
            config_fingerprint
        ),

        "selection_data": (
            "Official LIMUC train/validation "
            "partition only"
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,

        "checkpoint_selection": (
            "Validation patient QWK, then "
            "patient MAE, patient macro F1, "
            "and image QWK"
        ),

        "patient_reference_rule": (
            PRIMARY_REFERENCE_RULE
        ),

        "patient_prediction_aggregation": (
            "mean_probability"
        ),

        "oof_image_metrics": (
            oof_image_metrics
        ),

        "oof_patient_metrics": (
            oof_patient_metrics
        ),

        "oof_patient_95_ci": {
            "qwk": [
                qwk_ci_low,
                qwk_ci_high,
            ],
            "mae": [
                mae_ci_low,
                mae_ci_high,
            ],
            "macro_f1": [
                macro_f1_ci_low,
                macro_f1_ci_high,
            ],
            "accuracy": [
                accuracy_ci_low,
                accuracy_ci_high,
            ],
        },

        "oof_image_predictions_path": str(
            oof_image_path
        ),

        "oof_patient_predictions_path": str(
            oof_patient_path
        ),

        "bootstrap_ci_path": str(
            bootstrap_ci_path
        ),

        "bootstrap_samples_path": str(
            bootstrap_samples_path
        ),

        "fold_summary_path": str(
            fold_summary_path
        ),

        "pretrained_cfg_path": str(
            pretrained_cfg_path
        ),
    }

    summary_path = (
        save_dir
        /
        "summary.json"
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            summary,
            file,
            indent=2,
        )

    result_row = {
        "experiment_name": experiment_name,
        "display_name": preset["display_name"],
        "training_seed": int(training_seed),

        "analysis_plan_fingerprint": (
            analysis_plan_fingerprint
        ),
        "config_fingerprint": (
            config_fingerprint
        ),

        "loss_type": config["loss_type"],

        "ordinal_loss_weight": float(
            config[
                "ordinal_loss_weight"
            ]
        ),

        "ordinal_alpha": float(
            config["ordinal_alpha"]
        ),

        "oof_image_qwk": float(
            oof_image_metrics["qwk"]
        ),
        "oof_image_mae": float(
            oof_image_metrics["mae"]
        ),
        "oof_image_macro_f1": float(
            oof_image_metrics["macro_f1"]
        ),
        "oof_image_accuracy": float(
            oof_image_metrics["accuracy"]
        ),

        "oof_patient_qwk": float(
            oof_patient_metrics["qwk"]
        ),
        "oof_patient_qwk_ci_low": float(
            qwk_ci_low
        ),
        "oof_patient_qwk_ci_high": float(
            qwk_ci_high
        ),

        "oof_patient_mae": float(
            oof_patient_metrics["mae"]
        ),
        "oof_patient_mae_ci_low": float(
            mae_ci_low
        ),
        "oof_patient_mae_ci_high": float(
            mae_ci_high
        ),

        "oof_patient_macro_f1": float(
            oof_patient_metrics[
                "macro_f1"
            ]
        ),
        "oof_patient_macro_f1_ci_low": float(
            macro_f1_ci_low
        ),
        "oof_patient_macro_f1_ci_high": float(
            macro_f1_ci_high
        ),

        "oof_patient_accuracy": float(
            oof_patient_metrics[
                "accuracy"
            ]
        ),
        "oof_patient_accuracy_ci_low": float(
            accuracy_ci_low
        ),
        "oof_patient_accuracy_ci_high": float(
            accuracy_ci_high
        ),

        "official_test_loaded": False,
        "official_test_evaluated": False,

        "save_dir": str(save_dir),
    }

    with open(
        result_row_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            result_row,
            file,
            indent=2,
        )

    print("\nOOF image metrics:")
    print(
        json.dumps(
            oof_image_metrics,
            indent=2,
        )
    )

    print("\nOOF patient metrics:")
    print(
        json.dumps(
            oof_patient_metrics,
            indent=2,
        )
    )

    print("\nOOF patient bootstrap intervals:")
    display(bootstrap_ci_df)

    return result_row


# ============================================================
# 11. Execute all candidate losses
# ============================================================

result_rows = []

for training_seed in TRAIN_SEEDS:
    for experiment_name in EXPERIMENTS_TO_RUN:
        preset = EXPERIMENT_PRESETS[
            experiment_name
        ]

        result_row = run_experiment(
            experiment_name=(
                experiment_name
            ),
            preset=preset,
            training_seed=int(
                training_seed
            ),
        )

        result_rows.append(
            result_row
        )


# ============================================================
# 12. Save per-seed results
# ============================================================

results_by_seed_df = pd.DataFrame(
    result_rows
)

results_by_seed_path = (
    BASE_SAVE_ROOT
    /
    "vit_loss_screening_results_by_seed.csv"
)

results_by_seed_df.to_csv(
    results_by_seed_path,
    index=False,
)


# ============================================================
# 13. Aggregate results across training seeds
# ============================================================

group_columns = [
    "experiment_name",
    "display_name",
    "loss_type",
    "ordinal_loss_weight",
    "ordinal_alpha",
]

metric_columns = [
    "oof_image_qwk",
    "oof_image_mae",
    "oof_image_macro_f1",
    "oof_image_accuracy",

    "oof_patient_qwk",
    "oof_patient_mae",
    "oof_patient_macro_f1",
    "oof_patient_accuracy",
]

aggregation_specification = {}

for metric_name in metric_columns:
    aggregation_specification[
        f"{metric_name}_mean"
    ] = (
        metric_name,
        "mean",
    )

    aggregation_specification[
        f"{metric_name}_std"
    ] = (
        metric_name,
        "std",
    )

experiment_summary_df = (
    results_by_seed_df
    .groupby(
        group_columns,
        as_index=False,
    )
    .agg(
        n_training_seeds=(
            "training_seed",
            "nunique",
        ),
        **aggregation_specification,
    )
)

standard_deviation_columns = [
    column_name
    for column_name in (
        experiment_summary_df.columns
    )
    if column_name.endswith("_std")
]

experiment_summary_df[
    standard_deviation_columns
] = (
    experiment_summary_df[
        standard_deviation_columns
    ]
    .fillna(0.0)
)


# ============================================================
# 14. Development-only lexicographic ranking
# ============================================================

results_ranked_df = (
    experiment_summary_df
    .sort_values(
        by=[
            "oof_patient_qwk_mean",
            "oof_patient_mae_mean",
            "oof_patient_macro_f1_mean",
            "oof_image_qwk_mean",
        ],
        ascending=[
            False,
            True,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

results_ranked_df.insert(
    0,
    "development_rank",
    np.arange(
        1,
        len(results_ranked_df) + 1,
    ),
)

experiment_summary_path = (
    BASE_SAVE_ROOT
    /
    "vit_loss_screening_summary_by_experiment.csv"
)

results_ranked_path = (
    BASE_SAVE_ROOT
    /
    "vit_loss_screening_ranked_development_only.csv"
)

experiment_summary_df.to_csv(
    experiment_summary_path,
    index=False,
)

results_ranked_df.to_csv(
    results_ranked_path,
    index=False,
)


# ============================================================
# 15. Top-two shortlist
# ============================================================

shortlist_size = min(
    2,
    len(results_ranked_df),
)

vit_shortlist_df = (
    results_ranked_df
    .head(shortlist_size)
    .copy()
)

vit_shortlist_path = (
    BASE_SAVE_ROOT
    /
    "vit_loss_shortlist_for_multiseed_confirmation.csv"
)

vit_shortlist_df.to_csv(
    vit_shortlist_path,
    index=False,
)


# ============================================================
# 16. Display development-only ranking
# ============================================================

display_columns = [
    "development_rank",
    "experiment_name",
    "display_name",
    "n_training_seeds",

    "oof_patient_qwk_mean",
    "oof_patient_qwk_std",

    "oof_patient_mae_mean",
    "oof_patient_mae_std",

    "oof_patient_macro_f1_mean",
    "oof_patient_macro_f1_std",

    "oof_patient_accuracy_mean",
    "oof_patient_accuracy_std",

    "oof_image_qwk_mean",
    "oof_image_mae_mean",
]

print("\n" + "=" * 100)
print("DEVELOPMENT-ONLY VIT LOSS RANKING")
print("=" * 100)

display(
    results_ranked_df[
        display_columns
    ]
)

print("\n" + "=" * 100)
print("TOP-TWO VIT LOSS SHORTLIST")
print("=" * 100)

display(
    vit_shortlist_df[
        display_columns
    ]
)


# ============================================================
# 17. Display per-seed confidence intervals
# ============================================================

ci_display_columns = [
    "experiment_name",
    "training_seed",

    "oof_patient_qwk",
    "oof_patient_qwk_ci_low",
    "oof_patient_qwk_ci_high",

    "oof_patient_mae",
    "oof_patient_mae_ci_low",
    "oof_patient_mae_ci_high",

    "oof_patient_macro_f1",
    "oof_patient_macro_f1_ci_low",
    "oof_patient_macro_f1_ci_high",

    "oof_patient_accuracy",
    "oof_patient_accuracy_ci_low",
    "oof_patient_accuracy_ci_high",
]

print("\n" + "=" * 100)
print("PER-SEED OOF PATIENT METRICS WITH 95% BOOTSTRAP CIs")
print("=" * 100)

display(
    results_by_seed_df[
        ci_display_columns
    ]
    .sort_values(
        [
            "oof_patient_qwk",
            "oof_patient_mae",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


# ============================================================
# 18. Save compact screening summary
# ============================================================

screening_summary = {
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "analysis_plan_fingerprint": (
        analysis_plan_fingerprint
    ),

    "official_test_loaded": False,
    "official_test_evaluated": False,

    "primary_patient_reference": (
        PRIMARY_REFERENCE_RULE
    ),

    "patient_prediction_aggregation": (
        "mean_probability"
    ),

    "n_experiments": int(
        len(results_ranked_df)
    ),

    "n_training_seeds": int(
        results_by_seed_df[
            "training_seed"
        ].nunique()
    ),

    "top_ranked_screening_candidate": (
        results_ranked_df.iloc[0][
            "experiment_name"
        ]
    ),

    "shortlisted_candidates": (
        vit_shortlist_df[
            "experiment_name"
        ].tolist()
    ),

    "interpretation": (
        "The ranking is based on development OOF metrics only. "
        "The top-ranked single-seed candidate is not considered "
        "final. The top two candidates require multi-seed "
        "confirmation before cross-backbone comparison."
    ),

    "output_files": {
        "per_seed_results": str(
            results_by_seed_path
        ),
        "experiment_summary": str(
            experiment_summary_path
        ),
        "ranked_results": str(
            results_ranked_path
        ),
        "shortlist": str(
            vit_shortlist_path
        ),
    },
}

screening_summary_path = (
    BASE_SAVE_ROOT
    /
    "vit_loss_screening_summary.json"
)

with open(
    screening_summary_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        screening_summary,
        file,
        indent=2,
    )


# ============================================================
# 19. Final status
# ============================================================

print("\n" + "=" * 100)
print("CELL 3 COMPLETED SUCCESSFULLY")
print("=" * 100)

print(
    "Saved per-seed results to:",
    results_by_seed_path,
)

print(
    "Saved experiment summary to:",
    experiment_summary_path,
)

print(
    "Saved ranked development-only results to:",
    results_ranked_path,
)

print(
    "Saved top-two shortlist to:",
    vit_shortlist_path,
)

print(
    "Saved compact screening summary to:",
    screening_summary_path,
)

print(
    "\nOfficial LIMUC test data loaded:",
    False,
)

print(
    "Official LIMUC test data evaluated:",
    False,
)

print(
    "\nThe top-ranked candidate is a screening result, "
    "not the final model decision."
)